# Answer-flip study — revision run, Qwen3-VL-8B with V-CAST (Run-All)

This notebook adds **V-CAST** (NeurIPS 2026) at 5%, 10% and 25% of the visual tokens to the Qwen3-VL-8B replication, using the **official V-CAST code for Qwen3-VL**, on exactly the same 900 questions (300 each from Video-MME, EgoSchema and LongVideoBench). The four noise passes of these questions are already in the replication results, so only the full pass (to check that its answers match) and the three V-CAST passes are run.

| `BENCHMARK` | Questions | Expected time (T4 ×2) |
|---|---|---|
| `"videomme"` | 300 | about 1 h |
| `"egoschema"` | 300 | about 1 h |
| `"longvideobench"` | 300 | about 1–1.5 h |

## How to run
1. Settings → Accelerator **GPU T4 x2**, Internet **On**, HF_TOKEN in Add-ons → Secrets (same as before).
2. `BENCHMARK = "videomme"`, `QUICK_TEST = True` → **Run All** (~20 min). Check: `Compressor self-test OK ✅`, equivalence check `PASS`, `Cache check: PASS`, a few ✓ lines.
3. Then `QUICK_TEST = False` → **Save Version → Save & Run All (Commit)**, one benchmark per run.
4. Send back only `results.jsonl` from `flips_ext_qwen3vl_<benchmark>/` (and `errors.log` if it is not empty).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CONFIG — change only BENCHMARK and QUICK_TEST                    ║
# ╚══════════════════════════════════════════════════════════════════╝
BENCHMARK  = "videomme"   # one of: "videomme", "egoschema", "longvideobench"
QUICK_TEST = True         # True = ~30-min smoke test (real model, 4 questions). False = the real run.

MODEL = "qwen3vl"         # fixed for this notebook

# ── Replication sample: 300 questions per benchmark, drawn once (seed 20261015) from the main study's manifests.
#    Video-MME: 100 of the 213 confirmatory videos (all their questions); EgoSchema: 300 of 500;
#    LongVideoBench: 75 per duration group. The question IDs are fixed in REP_MANIFEST below (with their hash).
SAMPLE_SEED = 20261015
VIDEOMME_CONFIRM_CHUNKS = ["02", "08", "16", "18", "20"]
LVB_PER_GROUP = 125
MVBENCH_PER_TASK, MLVU_PER_TASK = 25, 72          # not used here (kept so the shared data code runs unchanged)
PREREG_URL = "https://claude.ai/code/artifact/65cb336e-3916-4101-9914-8dd1ca98e189"

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
NUM_FRAMES = 32                                     # as in the replication run (FlashVID's Qwen3-VL setting)
MIN_PIXELS, MAX_PIXELS = 50716, 200704              # per frame, as in the replication run
# Revision run: V-CAST (NeurIPS 2026) with its OFFICIAL Qwen3-VL code, on the same 900 questions as the replication.
METHOD_GRID = {"vcast": [0.05, 0.10, 0.25]}
VCAST = dict(min_k=1)                               # official defaults (temperature 0.7 is fixed inside the code)

# ── No new noise passes here: the four noise passes of these questions are already in the replication results ──
NOISE_FRAME_SHIFTS = {}
NOISE_OPTION_ORDERS = []

SEED = 0
MAX_RUNTIME_HOURS = 11.0
DATA_CHECK_ONLY = False
OUT_DIR  = f"/kaggle/working/flips_ext_{MODEL}_{BENCHMARK}"
WORK_DIR = "/tmp/astra_data"
QUICK_MAX_Q = 4 if QUICK_TEST else None
if QUICK_TEST:
    OUT_DIR = OUT_DIR + "_quick"
assert BENCHMARK in ("videomme", "egoschema", "longvideobench"), f"Unknown BENCHMARK {BENCHMARK}"
print("Model:", MODEL_ID, "| benchmark:", BENCHMARK, "| mode:", "QUICK TEST" if QUICK_TEST else "REAL RUN",
      "| frames:", NUM_FRAMES)

In [ ]:
# ── Frozen replication sample (question IDs per benchmark, with their SHA-256) ──
# Drawn once with seed 20261015 from the main study's question lists: Video-MME 100 of the 213 videos (all questions),
# EgoSchema 300 of 500 questions, LongVideoBench 75 questions per duration group.
REP_MANIFEST = {
    "videomme": {"sha256": "1f816d6cffb0ebb60c72f3e5252d25add7ecc6e58d6e5b1a7f71c0da45085977", "qids": ["014-1", "014-2", "014-3", "026-1", "026-2", "026-3", "032-1", "032-2", "032-3", "038-1", "038-2", "038-3", "066-1", "066-2", "066-3", "075-1", "075-2", "075-3", "126-1", "126-2", "126-3", "133-1", "133-2", "133-3", "163-1", "163-2", "163-3", "165-1", "165-2", "165-3", "180-1", "180-2", "180-3", "182-1", "182-2", "182-3", "188-1", "188-2", "188-3", "206-1", "206-2", "206-3", "210-1", "210-2", "210-3", "219-1", "219-2", "219-3", "224-1", "224-2", "224-3", "228-1", "228-2", "228-3", "240-1", "240-2", "240-3", "246-1", "246-2", "246-3", "258-1", "258-2", "258-3", "260-1", "260-2", "260-3", "266-1", "266-2", "266-3", "269-1", "269-2", "269-3", "270-1", "270-2", "270-3", "275-1", "275-2", "275-3", "283-1", "283-2", "283-3", "298-1", "298-2", "298-3", "323-1", "323-2", "323-3", "329-1", "329-2", "329-3", "332-1", "332-2", "332-3", "334-1", "334-2", "334-3", "335-1", "335-2", "335-3", "344-1", "344-2", "344-3", "349-1", "349-2", "349-3", "352-1", "352-2", "352-3", "355-1", "355-2", "355-3", "361-1", "361-2", "361-3", "362-1", "362-2", "362-3", "370-1", "370-2", "370-3", "380-1", "380-2", "380-3", "390-1", "390-2", "390-3", "394-1", "394-2", "394-3", "402-1", "402-2", "402-3", "410-1", "410-2", "410-3", "412-1", "412-2", "412-3", "419-1", "419-2", "419-3", "428-1", "428-2", "428-3", "429-1", "429-2", "429-3", "433-1", "433-2", "433-3", "437-1", "437-2", "437-3", "447-1", "447-2", "447-3", "449-1", "449-2", "449-3", "452-1", "452-2", "452-3", "456-1", "456-2", "456-3", "461-1", "461-2", "461-3", "463-1", "463-2", "463-3", "465-1", "465-2", "465-3", "466-1", "466-2", "466-3", "499-1", "499-2", "499-3", "502-1", "502-2", "502-3", "506-1", "506-2", "506-3", "507-1", "507-2", "507-3", "527-1", "527-2", "527-3", "534-1", "534-2", "534-3", "538-1", "538-2", "538-3", "547-1", "547-2", "547-3", "555-1", "555-2", "555-3", "556-1", "556-2", "556-3", "586-1", "586-2", "586-3", "606-1", "606-2", "606-3", "623-1", "623-2", "623-3", "674-1", "674-2", "674-3", "680-1", "680-2", "680-3", "705-1", "705-2", "705-3", "709-1", "709-2", "709-3", "713-1", "713-2", "713-3", "722-1", "722-2", "722-3", "724-1", "724-2", "724-3", "726-1", "726-2", "726-3", "744-1", "744-2", "744-3", "747-1", "747-2", "747-3", "761-1", "761-2", "761-3", "762-1", "762-2", "762-3", "768-1", "768-2", "768-3", "774-1", "774-2", "774-3", "779-1", "779-2", "779-3", "784-1", "784-2", "784-3", "794-1", "794-2", "794-3", "802-1", "802-2", "802-3", "838-1", "838-2", "838-3", "840-1", "840-2", "840-3", "856-1", "856-2", "856-3", "859-1", "859-2", "859-3", "862-1", "862-2", "862-3", "867-1", "867-2", "867-3", "871-1", "871-2", "871-3", "875-1", "875-2", "875-3", "879-1", "879-2", "879-3", "894-1", "894-2", "894-3"]},
    "egoschema": {"sha256": "c230cd8256575f5b036bd3ddfaa1bc85e4fd740058c71caf0e926db77b0f9469", "qids": ["00000", "00001", "00002", "00003", "00004", "00005", "00006", "00007", "00008", "00009", "00010", "00011", "00013", "00014", "00017", "00019", "00021", "00024", "00026", "00027", "00028", "00029", "00030", "00031", "00032", "00033", "00035", "00036", "00038", "00039", "00040", "00041", "00042", "00043", "00044", "00046", "00049", "00053", "00054", "00056", "00058", "00059", "00061", "00064", "00065", "00066", "00068", "00069", "00071", "00072", "00073", "00074", "00075", "00076", "00077", "00079", "00080", "00083", "00084", "00085", "00087", "00088", "00092", "00093", "00095", "00096", "00097", "00098", "00099", "00101", "00102", "00104", "00105", "00106", "00107", "00114", "00115", "00116", "00117", "00119", "00120", "00121", "00123", "00124", "00125", "00126", "00127", "00128", "00130", "00134", "00135", "00136", "00138", "00140", "00141", "00142", "00143", "00144", "00145", "00146", "00149", "00150", "00151", "00152", "00153", "00154", "00155", "00156", "00159", "00160", "00161", "00162", "00164", "00166", "00167", "00168", "00172", "00173", "00174", "00176", "00178", "00182", "00187", "00188", "00189", "00190", "00191", "00192", "00194", "00200", "00201", "00203", "00204", "00206", "00209", "00210", "00211", "00212", "00214", "00220", "00222", "00223", "00227", "00228", "00231", "00232", "00234", "00235", "00237", "00238", "00239", "00240", "00241", "00242", "00245", "00246", "00247", "00248", "00250", "00251", "00252", "00253", "00254", "00255", "00257", "00258", "00259", "00260", "00262", "00263", "00264", "00265", "00266", "00267", "00270", "00271", "00273", "00274", "00277", "00278", "00280", "00281", "00284", "00285", "00287", "00288", "00289", "00290", "00292", "00294", "00295", "00296", "00297", "00298", "00300", "00302", "00305", "00308", "00310", "00313", "00314", "00315", "00317", "00318", "00320", "00321", "00322", "00324", "00328", "00331", "00333", "00334", "00335", "00338", "00341", "00342", "00343", "00344", "00346", "00349", "00351", "00352", "00355", "00356", "00357", "00358", "00359", "00360", "00362", "00363", "00364", "00365", "00366", "00368", "00369", "00371", "00377", "00379", "00380", "00384", "00386", "00390", "00392", "00394", "00395", "00397", "00398", "00399", "00400", "00402", "00404", "00405", "00406", "00408", "00409", "00414", "00416", "00418", "00419", "00420", "00421", "00422", "00423", "00425", "00426", "00428", "00429", "00433", "00436", "00437", "00438", "00439", "00441", "00445", "00447", "00450", "00451", "00457", "00460", "00465", "00467", "00471", "00473", "00474", "00475", "00476", "00479", "00480", "00481", "00483", "00484", "00485", "00487", "00488", "00489", "00492", "00494", "00495", "00497", "00499"]},
    "longvideobench": {"sha256": "59e138d85b68b51a9509a7a4c73b0b1f0b2c0868f4ab23446db35b39c86f5dec", "qids": ["-Xg2-SUq6wo_0", "-Xg2-SUq6wo_1", "-if_Wg43vTk_0", "-kaF6SnSEo8_0", "0WEnmqVVbHo_0", "0WEnmqVVbHo_1", "0ln2qdCR5lA_0", "0mEN5Jf2hU0_0", "0mEN5Jf2hU0_1", "0rWA-p4p5IM_1", "14ot4DrXdds_0", "1R5uPaL0V-0_1", "1oL4ugMdDvQ_0", "1pxrIj9Xyps_1", "2F7d7aUCmUU_0", "2W2ZkYARds4_1", "2vVQo_GMA70_1", "2zZSMnGLGao_1", "3Qv2K_0GEE0_0", "3avWNHoEDAg_0", "3hyPwjkdHEA_1", "3t_Knk7FWT8_0", "3u__SZlBLC0_0", "49YMA0f1yhI_0", "4ouAf1ldH60_2", "58mCJVNMJvw_0", "5zbV24vyO44_1", "6d-EVupvWzU_1", "6moss7hGbvg_0", "7F9IrtSHmc0_1", "7TljSpTBS9c_0", "88LbBgZP1vQ_1", "8MkL3W6wU3g_2", "8Qe03WDCrB4_0", "8hhcFRoR0mw_1", "8wbNEJjBa0k_0", "9PD3ciudpIE_1", "9aWPbYosJUw_0", "9yQV0t5meo4_0", "@_eat_sleep_travel_repeat-7296468146473536800_0", "@healthfood-6874943613630041350_0", "@healthfood-6948509102305791238_0", "@healthfood-7115052132578921774_0", "@healthfood-7125042270381853995_0", "@healthfood-7135100078221430058_0", "@healthfood-7200404243168415022_0", "@jess.morg-7195624647830850858_0", "@jess.morg-7255843192677911851_0", "@jess.morg-7275475379375164715_0", "@jess.morg-7291800958152084782_0", "@jetset_anna-6933580627824495878_0", "@jetset_anna-7126584501228129542_0", "@jonijawne-7184469162956246277_0", "@jonijawne-7197144530024500485_0", "@jonijawne-7214559428471770374_0", "@jonijawne-7305901654417706246_0", "@jonijawne-7312063059160206597_0", "@kelseyinlondon-7094322812327906565_0", "@kerstinong-6900436106790194434_0", "@kerstinong-7197781091162279169_0", "@kerstinong-7254520467002854657_0", "@lisolna-7203500499360877829_0", "@lisolna-7235995901477604635_0", "@lisolna-7282789187676294432_0", "@lisolna-7288701821369978144_0", "@lisolna-7321010343067618592_0", "@lisolna-7359951777845775648_0", "@luxtravelbe-7233851486474636570_0", "@luxtravelbe-7286546546932370721_0", "@movie.explained6-7253164232307379457_0", "@movie.explained6-7253792552996867330_0", "@movie.explained6-7254803496900267266_0", "@movie.explained6-7257359603489443073_0", "@movie.explained6-7259227637992705282_0", "@movie.explained6-7262938043315686664_0", "@movie.explained6-7268936523481943297_0", "@movie.explained6-7270293967017725185_0", "@movie.explained6-7275653401025826050_0", "@movie.explained6-7277140989259599105_0", "@movie.explained6-7277548218106367234_0", "@movie.explained6-7278310085217094914_0", "@placesunleashed-7289528053112343814_0", "@placesunleashed-7297000148201262341_0", "@placesunleashed-7303594391850044678_0", "@placesunleashed-7308098412078025989_0", "@placesunleashed-7324498849857326341_0", "@placesunleashed-7337436458078260485_0", "@recipesbyanne-7141686631676808454_0", "@recipesbyanne-7153617149041446150_0", "@recipesbyanne-7161798161395240197_0", "@recipesbyanne-7231566545263103259_0", "@thatrecipe.us-7266555380925287723_0", "@thatrecipe.us-7301855783438896427_0", "@thatrecipe.us-7335998138500648235_0", "@tiffycooks-6940704678569053446_0", "@tiffycooks-7062861483687873798_0", "@tiffycooks-7080578381569445125_0", "@tiffycooks-7171182579058511110_0", "AHeq99pojLo_0", "Aau-XoIebno_2", "BtaVRhoLpC0_0", "CEZ9rbjK3P4_0", "DVsw1brd_Yc_0", "DX-mYMYf8jI_0", "Dkm35G5kkcc_0", "DoizYSYQRqU_2", "E7FSg22MdKE_1", "ErGYJ7kqIow_0", "F2OhCCEIOcU_0", "F2OhCCEIOcU_1", "F2i-LchFVCU_0", "F4bDyyEO4PU_0", "F4bDyyEO4PU_1", "F8Ma1qs0Rkg_1", "Fq3zbbp-lv4_1", "GAoK4XjssrM_0", "GHScTyR2OgQ_0", "GZFL58_pXPg_0", "GwcXYTX2GmA_0", "H2ksp6sRR-k_2", "H7gnsO4UZmU_0", "HAED3riiZkw_0", "HPunfsyjETs_0", "HWyDOQrYtCk_0", "I2Fyzav8MxE_0", "IN0osLg-Mn8_1", "Ip9DbdOtqF4_1", "J1IwKg2ufk8_2", "JIoE18JYGcM_0", "J_ZmaKRpyoU_2", "Jfp1Ks7Hh1E_1", "Jfp1Ks7Hh1E_2", "Jim_NG0PYpc_0", "JpjytHmGHZ4_0", "LAQRjgx_OY8_1", "LYvStKy8iAc_1", "Lc7RikDaa30_0", "LcZxjqtzXJI_1", "Lf7uW4xK2bc_0", "LfUsGv-ESbc_1", "LlqsCCa6y58_2", "M-YfPangEfA_0", "M5Umm-ltnao_0", "M7YSCIkUaNw_1", "MJYBHfYF8LI_1", "MTmUo6QITo4_0", "Msz128EJeWE_0", "MvgJAD6tZXo_2", "MyDAFlHsYsQ_0", "NHIT9vq6mJU_1", "Ng2rNm6Nwsg_0", "NpYUxd1vUUE_2", "O471uwTNx6k_2", "P0BSTjziVys_0", "PE32sjfN-uM_0", "PbiTIR8N4Hc_1", "PzUxuZ-KGsU_0", "QHS9ZZBdK-g_0", "QPth_xqBXGY_2", "RTUFPjliMCU_0", "SYqEMs0EYoI_0", "Sn7JPKbG6tY_2", "T15Kv6dtYO0_0", "T15Kv6dtYO0_2", "TMe7oXMJoSM_2", "TXiX3NO5f5w_0", "TlaX2iIYZD4_1", "TynnVVRNZGs_0", "UD5ifzOPzhc_0", "UN3ICsfqKEY_1", "UbNyMSwoT5A_0", "UbgwG8fcIu0_0", "UwJTCg5fpXg_1", "Uy_o-WCq2Cc_0", "V0h7rJShw0g_0", "VLInjyogciw_0", "VNvH5u5CWV8_0", "VPcdtoldWCA_0", "VkNF0rXuDXw_1", "Vn-z0qXp9bQ_0", "VwZeSoYugZk_1", "WHxBO7XecSY_0", "WaiGdRYD36k_1", "X5v4nBo5y28_2", "XJ6REZOXsvM_0", "XVXczyheik0_1", "Y1YCvEip_ko_2", "YcbKamVxDzI_1", "YcbKamVxDzI_2", "YehqA9xoGTY_1", "YiHjsOqxtk0_0", "Z6Hx_BAZeUw_0", "Z6Hx_BAZeUw_2", "Z7Cox6lPW3c_1", "ZoUsR8t8IxE_0", "ZoUsR8t8IxE_1", "_EUDpS9UF9o_0", "aVHAr8rc-Ks_1", "aX_HgA5SNLQ_0", "akoJDx23QWU_0", "anQ5KFW8Gn4_0", "aqUisZS9Ruw_0", "aqUisZS9Ruw_1", "bDpgz_2Piqg_0", "bH1Q9AM5vik_0", "baFsMWNavQ4_0", "d8H7hgQY9ew_0", "d8H7hgQY9ew_1", "dE5iWeCVpGI_0", "e-EtQHH4aWE_0", "e11Q4ThFu5A_0", "eE5Z7gDbgVA_1", "e_nAKiSutiA_0", "er1oRjH2iu8_1", "fO7nwCix8xU_0", "fPLjjr8w6DU_1", "fZBC3nmvJb8_0", "fryyNwUCPWA_1", "fvCrE5NCsts_1", "g7zuBUMBr2E_0", "gAgCnu82RHE_1", "gJuOMPiixUA_0", "gtX_oRpLClY_0", "gyV6EqgiPNg_0", "h0OHi9uAcBo_0", "h4jIoMxZopU_0", "h4jIoMxZopU_1", "hf4WUOagFAw_1", "hznvV2bBkX4_0", "i6C6r2g4Y7Q_0", "i8TJ7RgimNM_0", "iHNjWhx3EaI_0", "iJgh2dnudIU_1", "ie9NxdgNe0g_0", "iwXp1fT89-M_1", "iwXp1fT89-M_2", "ixAU3l0sX_o_0", "jJGbXCCU5yc_0", "jbPR2SJuFHg_0", "jvkmcX47bKU_1", "jvkmcX47bKU_2", "k194yU67Zik_0", "k4jiEuZbN-4_0", "kFHVBCEwC3w_0", "kWUmHAzCp7s_0", "kbRtl58u_kk_0", "l3vxOGgAM2g_0", "lN3WnXMaE0o_0", "lN3WnXMaE0o_1", "lNReCCShKJQ_0", "lcEaHk8f4Co_0", "luRqMb5qfhM_0", "m1mGerQhdHA_0", "m7sL8LuMD8Q_0", "mB7NW91REF4_0", "mFcEWmtn3ag_1", "mHARxee4EzQ_0", "mS1QPVgBDQo_1", "mS1QPVgBDQo_2", "mTn_C-SyW84_0", "mbcvVYobCXI_1", "mld0TnA2jEs_1", "nBWB-h68E4k_0", "nvG2YMWiwqU_0", "o2F-N42Ufo4_2", "oCXKARwr6PA_0", "oO8OJZFZT9s_0", "oddHY1vwcjo_0", "pFtKaT3GF9I_0", "pPJq1rMDRGs_1", "q3FAxTSENEw_1", "qZVBFAtfp2A_0", "qbA42wQoWAs_0", "qksR2Zvd-FM_2", "qsH6q5wNso4_1", "rP7sQe784k8_1", "rwL_XPw46zQ_1", "rzIiQ4Vxlbk_0", "t1nhAnMQBHg_0", "t1nhAnMQBHg_1", "t3P11ENBZyc_1", "tdA5atpqaAc_0", "u1D4ArcBjLI_1", "u5NAcHhI_Uc_0", "uDZrl35yeZE_0", "v-7zF3Y0yJs_0", "v0QLje6xYgA_0", "vJ9hYCUDHTo_0", "vJ9hYCUDHTo_1", "vVRC-0VKPrg_2", "vix2_R79-l4_0", "wGWbI3BHVM8_0", "wSHPuI7wWIg_2", "xSzXFQPldJg_0", "yFAuXmcGk2Y_0", "yqejTvYILlA_2", "ysRFFN5nzqE_2", "yz3lOAe32Tw_0", "z8YF-D-t7AI_0", "zVudr8cxHRE_1"]},
}
print({b: len(v["qids"]) for b, v in REP_MANIFEST.items()}, "questions per benchmark")

### 0 · Reference answers of the replication run (for the determinism check)

In [ ]:
# ── Reference: the full model's answers and option log-probabilities from the replication run (released data) ──
# The revision run recomputes the full pass; cell 7 compares it with this reference.
REFERENCE_FULL = {"egoschema":{"00000":["D",-22.094,-22.297,-15.531,-0.0,-21.438],"00001":["A",-0.618,-11.649,-12.806,-11.415,-0.774],"00002":["B",-2.073,-0.135,-9.01,-10.198,-7.807],"00003":["E",-9.257,-1.601,-8.694,-13.273,-0.226],"00004":["D",-10.675,-10.894,-2.238,-0.113,-11.519],"00005":["E",-16.969,-17.438,-17.688,-17.625,0.0],"00006":["B",-7.534,-0.002,-6.346,-8.44,-11.549],"00007":["E",-15.257,-0.741,-13.46,-9.913,-0.647],"00008":["D",-4.014,-6.123,-3.983,-0.077,-3.358],"00009":["A",-0.006,-6.818,-12.021,-12.006,-5.412],"00010":["E",-3.019,-10.831,-11.222,-12.613,-0.05],"00011":["E",-11.438,-8.969,-11.625,-12.516,-0.0],"00013":["A",-0.085,-5.757,-3.304,-3.179,-12.835],"00014":["A",-0.0,-12.812,-16.594,-18.0,-19.422],"00017":["D",-16.719,-15.875,-15.719,-0.0,-19.688],"00019":["B",-12.204,-0.001,-6.782,-12.204,-12.986],"00021":["A",0.0,-18.844,-20.578,-19.844,-22.297],"00024":["D",-10.311,-8.404,-1.857,-0.17,-13.061],"00026":["E",-2.101,-6.695,-9.523,-9.898,-0.132],"00027":["E",-8.219,-14.235,-15.813,-15.031,-0.0],"00028":["C",-10.304,-3.476,-0.32,-4.523,-1.461],"00029":["B",-0.939,-0.502,-5.689,-11.033,-10.205],"00030":["C",-15.906,-16.875,-0.0,-17.75,-14.969],"00031":["E",-0.809,-9.965,-9.309,-9.371,-0.59],"00032":["C",-14.061,-2.155,-0.123,-10.076,-11.998],"00033":["A",-0.0,-13.781,-12.562,-15.391,-13.688],"00035":["A",-0.001,-8.064,-7.532,-8.47,-8.783],"00036":["E",-3.14,-4.953,-2.234,-9.484,-0.171],"00038":["B",-5.642,-0.048,-12.595,-3.142,-13.001],"00039":["A",-0.004,-6.629,-6.004,-11.285,-11.113],"00040":["A",-0.001,-11.844,-12.688,-13.719,-7.376],"00041":["C",-16.672,-15.859,-0.0,-17.281,-16.234],"00042":["E",-4.228,-3.712,-0.884,-7.368,-0.603],"00043":["C",-19.288,-2.21,-0.116,-15.835,-18.944],"00044":["B",-11.489,-0.036,-10.817,-3.348,-14.989],"00046":["B",-16.922,-0.0,-15.953,-9.688,-15.641],"00049":["D",-14.442,-11.661,-5.348,-0.005,-12.833],"00053":["C",-5.411,-1.473,-0.379,-8.27,-2.504],"00054":["D",-22.281,-19.469,-21.5,0.0,-25.641],"00056":["A",-0.0,-8.719,-9.875,-13.391,-9.781],"00058":["E",-17.391,-19.578,-19.172,-19.281,0.0],"00059":["C",-12.375,-8.375,-0.0,-10.156,-15.266],"00061":["E",-12.9,-7.119,-10.166,-0.775,-0.619],"00064":["E",-5.007,-9.944,-13.382,-15.085,-0.007],"00065":["B",-16.281,-0.0,-16.531,-16.844,-19.172],"00066":["A",-0.0,-15.562,-16.562,-15.875,-12.812],"00068":["A",-0.12,-7.308,-2.183,-8.683,-18.386],"00069":["E",-8.444,-2.194,-9.804,-0.944,-0.694],"00071":["B",-20.109,-0.0,-13.344,-17.859,-19.156],"00072":["C",-21.594,-20.828,0.0,-19.75,-20.938],"00073":["E",-15.672,-12.453,-16.734,-9.875,-0.0],"00074":["A",-0.068,-7.396,-3.006,-10.709,-4.146],"00075":["D",-16.938,-16.531,-17.406,-0.0,-19.938],"00076":["B",-12.82,-0.007,-8.57,-4.976,-12.726],"00077":["E",-4.204,-6.891,-8.673,-9.032,-0.016],"00079":["A",-0.357,-9.514,-1.326,-3.357,-15.951],"00080":["B",-8.319,-0.007,-5.007,-14.476,-16.538],"00083":["B",-20.688,0.0,-16.938,-18.781,-18.906],"00084":["E",-6.096,-12.612,-16.112,-16.518,-0.002],"00085":["D",-13.859,-9.031,-15.266,-0.0,-13.266],"00087":["B",-19.156,-0.0,-18.172,-14.75,-20.5],"00088":["C",-13.282,-7.938,-0.001,-7.485,-14.392],"00092":["A",-0.252,-1.72,-5.314,-8.111,-3.267],"00093":["E",-17.609,-15.094,-14.781,-15.281,-0.0],"00095":["B",-5.597,-0.004,-11.676,-10.535,-14.644],"00096":["C",-7.188,-9.845,-0.001,-11.563,-15.391],"00097":["E",-8.059,-6.95,-4.575,-2.591,-0.09],"00098":["C",-15.797,-14.75,-0.0,-15.875,-17.047],"00099":["E",-5.722,-9.285,-13.128,-15.769,-0.003],"00101":["C",-13.656,-13.062,-0.0,-11.5,-14.438],"00102":["D",-2.488,-0.863,-6.394,-0.707,-10.472],"00104":["B",-1.64,-0.265,-3.734,-4.203,-11.171],"00105":["C",-13.828,-12.469,-0.0,-12.5,-13.422],"00106":["E",-8.379,-7.582,-8.363,-5.832,-0.004],"00107":["A",-0.109,-12.578,-2.266,-14.328,-14.438],"00114":["B",-8.682,-0.042,-11.292,-3.198,-10.198],"00115":["C",-6.034,-8.878,-0.003,-9.721,-16.409],"00116":["B",-17.75,-0.0,-15.719,-16.531,-16.688],"00117":["E",-20.75,-19.344,-19.281,-19.625,0.0],"00119":["E",-10.828,-8.813,-9.5,-10.078,-0.0],"00120":["C",-14.922,-8.032,-0.0,-13.703,-11.125],"00121":["A",-0.259,-1.962,-5.494,-3.165,-3.181],"00123":["D",-5.236,-2.892,-9.658,-0.08,-4.142],"00124":["D",-11.904,-7.467,-1.779,-0.186,-10.498],"00125":["C",-10.55,-6.346,-0.003,-7.065,-11.034],"00126":["E",-9.395,-8.208,-10.598,-5.504,-0.004],"00127":["B",-18.188,0.0,-17.719,-18.344,-20.719],"00128":["D",-24.0,-22.906,-21.844,0.0,-22.281],"00130":["A",-0.064,-10.876,-2.783,-13.923,-12.908],"00134":["C",-3.627,-4.487,-0.221,-5.909,-1.846],"00135":["B",-13.297,-0.0,-12.047,-10.844,-17.109],"00136":["B",-18.077,-0.046,-15.264,-17.827,-3.108],"00138":["A",-0.015,-4.702,-5.171,-11.046,-11.671],"00140":["B",-13.812,-0.0,-11.875,-13.703,-16.281],"00141":["B",-2.798,-0.47,-1.783,-2.033,-4.189],"00142":["B",-9.394,-0.191,-9.378,-1.753,-13.05],"00143":["B",-1.338,-0.307,-6.713,-8.525,-8.541],"00144":["D",-12.562,-13.547,-13.344,-0.0,-11.531],"00145":["B",-19.828,-0.0,-15.031,-17.469,-19.891],"00146":["C",-15.824,-2.637,-0.074,-12.371,-11.137],"00149":["E",-15.5,-15.656,-10.812,-12.375,-0.0],"00150":["A",-0.009,-9.494,-5.853,-11.025,-5.041],"00151":["A",-0.0,-10.719,-10.656,-13.172,-14.281],"00152":["B",-4.914,-0.007,-11.554,-13.179,-13.929],"00153":["B",-2.477,-0.102,-4.321,-12.102,-13.305],"00154":["C",-6.018,-3.955,-0.096,-2.674,-6.862],"00155":["B",-2.784,-0.065,-6.456,-11.784,-15.472],"00156":["D",-15.303,-14.584,-15.787,-0.006,-5.131],"00159":["E",-11.117,-10.633,-1.18,-9.867,-0.367],"00160":["A",-0.476,-4.132,-4.788,-1.038,-7.898],"00161":["E",-10.59,-2.465,-8.449,-7.137,-0.09],"00162":["B",-2.931,-0.056,-7.431,-10.227,-12.915],"00164":["B",-6.893,-0.268,-1.455,-7.955,-14.486],"00166":["D",-7.469,-14.86,-17.688,-0.001,-17.422],"00167":["E",-15.368,-1.336,-12.055,-13.852,-0.305],"00168":["B",-2.128,-0.128,-6.597,-16.425,-14.941],"00172":["B",-14.531,-0.0,-15.125,-8.313,-18.5],"00173":["D",-2.397,-5.022,-4.475,-0.116,-9.287],"00174":["B",-14.641,-0.0,-15.312,-15.266,-10.938],"00176":["D",-10.613,-9.769,-9.378,-0.003,-5.722],"00178":["A",-0.331,-10.299,-10.565,-1.268,-10.784],"00182":["E",-4.511,-12.761,-11.761,-12.042,-0.011],"00187":["C",-2.753,-13.566,-0.066,-13.378,-13.628],"00188":["B",-2.792,-0.698,-2.885,-0.979,-4.651],"00189":["A",-0.0,-17.953,-18.906,-19.422,-16.625],"00190":["E",-5.081,-0.956,-4.269,-11.269,-0.519],"00191":["E",-17.656,-19.656,-20.031,-22.75,0.0],"00192":["C",-11.859,-11.828,-0.0,-13.828,-11.516],"00194":["A",-0.021,-5.708,-7.052,-4.349,-5.677],"00200":["E",-7.689,-7.017,-0.939,-6.111,-0.502],"00201":["A",-0.001,-6.939,-12.142,-13.923,-8.376],"00203":["E",-11.955,-2.127,-10.236,-10.721,-0.127],"00204":["B",-7.698,-0.01,-6.51,-11.604,-4.792],"00206":["D",-21.719,-14.844,-22.625,-0.0,-22.219],"00209":["A",-0.004,-5.567,-9.113,-9.207,-15.738],"00210":["C",-12.281,-10.859,-0.0,-13.609,-14.016],"00211":["B",-14.082,-0.097,-2.379,-10.379,-13.191],"00212":["C",-8.749,-3.983,-0.03,-8.124,-4.561],"00214":["C",-12.354,-0.791,-0.604,-11.338,-11.244],"00220":["B",-16.562,-0.0,-19.406,-19.969,-22.203],"00222":["D",-17.594,-10.031,-18.938,-0.0,-21.109],"00223":["A",-0.0,-15.25,-12.625,-15.188,-16.141],"00227":["B",-9.058,-0.152,-4.714,-2.027,-9.792],"00228":["A",-0.259,-1.478,-10.509,-8.369,-12.056],"00231":["D",-4.398,-7.742,-5.836,-0.023,-4.914],"00232":["E",-11.11,-9.422,-9.344,-10.36,-0.0],"00234":["C",-8.344,-11.281,-0.0,-11.656,-10.266],"00235":["E",-18.344,-19.172,-20.438,-19.859,0.0],"00237":["A",-0.0,-13.656,-13.469,-11.938,-8.656],"00238":["E",-26.312,-25.5,-25.906,-26.031,0.0],"00239":["B",-14.849,-0.005,-12.927,-5.224,-12.708],"00240":["A",-0.225,-1.6,-14.304,-13.96,-10.475],"00241":["B",-9.894,-0.004,-5.785,-8.035,-15.097],"00242":["C",-5.387,-6.496,-0.043,-3.902,-4.152],"00245":["B",-3.494,-0.431,-3.806,-9.29,-1.212],"00246":["A",-0.0,-12.391,-13.125,-13.172,-14.609],"00247":["A",-0.001,-12.798,-14.142,-6.751,-13.267],"00248":["D",-13.893,-10.096,-6.252,-0.002,-11.189],"00250":["E",-10.031,-15.281,-18.516,-15.672,-0.0],"00251":["C",-6.835,-1.992,-0.148,-9.398,-16.507],"00252":["C",-16.172,-14.547,-0.0,-15.953,-14.219],"00253":["E",-2.018,-7.237,-12.237,-11.956,-0.143],"00254":["A",-0.695,-6.289,-0.695,-9.054,-10.258],"00255":["A",-0.001,-7.813,-12.501,-7.751,-16.376],"00257":["D",-17.281,-18.844,-20.547,-0.0,-17.219],"00258":["C",-18.406,-18.703,-0.0,-17.375,-16.438],"00259":["A",-0.0,-10.375,-10.407,-9.438,-8.735],"00260":["E",-13.777,-12.527,-11.512,-2.871,-0.058],"00262":["E",-17.375,-17.359,-14.312,-14.312,-0.0],"00263":["D",-9.385,-4.573,-10.589,-0.01,-10.62],"00264":["D",-5.543,-1.184,-9.559,-0.371,-13.356],"00265":["A",-0.0,-16.531,-18.672,-16.375,-17.594],"00266":["E",-15.906,-16.547,-9.969,-14.234,-0.0],"00267":["C",-9.069,-6.538,-0.429,-4.679,-1.085],"00270":["A",-0.002,-6.502,-10.002,-9.627,-12.97],"00271":["E",-12.703,-14.656,-15.516,-14.688,-0.0],"00273":["C",-5.691,-2.503,-0.378,-7.722,-1.472],"00274":["C",-12.844,-13.531,-0.0,-12.156,-13.406],"00277":["C",-13.954,-7.095,-0.001,-14.001,-9.063],"00278":["B",-10.0,-0.0,-12.156,-13.406,-11.813],"00280":["B",-16.969,-0.0,-15.969,-14.188,-13.562],"00281":["A",-0.0,-17.375,-17.031,-15.5,-17.188],"00284":["B",-3.7,-0.138,-13.013,-2.263,-13.997],"00285":["B",-9.386,-0.058,-8.23,-4.683,-3.058],"00287":["C",-14.063,-15.61,-0.0,-12.094,-8.438],"00288":["D",-10.321,-4.946,-6.821,-0.008,-10.915],"00289":["A",-0.0,-16.375,-14.656,-16.656,-18.156],"00290":["B",-20.188,0.0,-18.938,-17.781,-20.25],"00292":["E",-3.657,-2.97,-10.313,-4.345,-0.094],"00294":["C",-4.263,-2.419,-0.138,-9.95,-3.669],"00295":["B",-12.54,-0.134,-3.509,-8.884,-2.352],"00296":["A",-0.09,-2.497,-6.997,-5.715,-14.012],"00297":["E",-13.297,-8.75,-10.375,-14.953,-0.0],"00298":["A",-0.005,-5.318,-7.943,-11.302,-13.146],"00300":["B",-5.263,-0.638,-8.997,-9.466,-0.763],"00302":["D",-15.188,-12.375,-13.062,-0.0,-10.344],"00305":["E",-6.089,-2.948,-7.073,-2.323,-0.167],"00308":["A",-0.135,-8.541,-11.072,-2.072,-14.51],"00310":["A",-0.0,-14.5,-17.188,-18.219,-18.562],"00313":["C",-20.531,-18.859,0.0,-18.969,-20.672],"00314":["B",-10.549,-0.44,-13.877,-1.033,-13.549],"00315":["E",-10.016,-7.626,-12.266,-14.422,-0.001],"00317":["B",-17.161,-0.005,-13.692,-5.38,-15.161],"00318":["B",-4.104,-0.041,-3.729,-10.401,-9.792],"00320":["D",-13.734,-9.156,-14.609,-0.0,-9.875],"00321":["D",-1.466,-1.216,-9.529,-0.998,-2.263],"00322":["B",-17.109,-0.0,-15.906,-16.375,-16.672],"00324":["E",-5.858,-3.17,-5.342,-4.686,-0.061],"00328":["D",-17.219,-16.219,-14.5,-0.0,-19.312],"00331":["A",-0.128,-2.535,-3.191,-12.097,-13.691],"00333":["B",-19.031,0.0,-17.828,-17.156,-20.391],"00334":["C",-18.922,-17.078,-0.0,-16.734,-19.469],"00335":["B",-15.565,-0.003,-12.675,-5.847,-14.128],"00338":["C",-16.766,-8.281,-0.0,-15.688,-17.469],"00341":["A",-0.016,-8.61,-4.454,-5.454,-12.954],"00342":["B",-9.801,-0.004,-10.238,-10.52,-5.566],"00343":["A",-0.0,-15.844,-15.438,-17.703,-18.0],"00344":["D",-25.234,-23.719,-22.75,0.0,-21.469],"00346":["A",-0.008,-5.352,-10.946,-8.883,-5.696],"00349":["C",-17.031,-16.391,-0.0,-16.328,-15.953],"00351":["B",-2.867,-0.21,-7.835,-9.382,-2.023],"00352":["E",-1.748,-2.123,-1.467,-6.107,-0.748],"00355":["D",-2.306,-4.416,-9.025,-0.119,-9.431],"00356":["C",-4.211,-5.117,-0.023,-9.617,-6.086],"00357":["C",-21.609,-11.219,-0.0,-15.562,-19.047],"00358":["E",-14.562,-16.922,-13.75,-17.656,-0.0],"00359":["E",-10.837,-10.899,-9.977,-4.79,-0.008],"00360":["B",-18.438,-0.0,-13.406,-15.953,-15.75],"00362":["E",-14.109,-12.625,-15.906,-14.047,-0.0],"00363":["A",-0.041,-7.978,-4.916,-3.447,-8.275],"00364":["C",-16.016,-15.953,-0.0,-16.719,-18.453],"00365":["C",-14.483,-9.592,-0.201,-12.686,-1.702],"00366":["E",-6.19,-11.346,-9.846,-12.143,-0.002],"00368":["A",-0.0,-10.313,-11.047,-12.109,-11.766],"00369":["B",-2.697,-0.104,-3.478,-9.807,-11.26],"00371":["D",-12.567,-10.833,-3.896,-0.021,-13.661],"00377":["A",-0.001,-8.032,-9.032,-11.282,-11.266],"00379":["E",-6.701,-0.936,-5.592,-4.295,-0.529],"00380":["E",-15.062,-11.062,-16.844,-17.984,-0.0],"00384":["C",-14.25,-14.906,-0.0,-17.203,-14.797],"00386":["D",-16.5,-19.906,-19.906,-0.0,-13.312],"00390":["D",-7.844,-11.813,-15.797,-0.0,-18.235],"00392":["C",-9.348,-10.207,-0.004,-5.535,-13.488],"00394":["E",-3.897,-8.053,-6.959,-9.803,-0.022],"00395":["C",-7.524,-4.977,-0.008,-13.29,-6.946],"00397":["A",-0.007,-9.398,-8.116,-12.663,-5.038],"00398":["D",-21.156,-20.469,-20.969,0.0,-21.406],"00399":["A",-0.0,-15.781,-10.5,-17.281,-17.406],"00400":["B",-6.248,-0.592,-10.483,-0.811,-8.795],"00402":["E",-3.17,-7.201,-7.076,-13.748,-0.045],"00404":["D",-7.818,-10.693,-2.99,-0.052,-7.724],"00405":["C",-1.504,-4.348,-0.317,-3.317,-12.113],"00406":["D",-11.938,-13.469,-13.656,-0.0,-16.875],"00408":["B",-5.192,-0.036,-9.129,-10.77,-3.536],"00409":["A",-0.0,-12.406,-12.094,-13.859,-12.547],"00414":["C",-11.812,-12.687,-0.109,-6.015,-2.296],"00416":["A",-0.001,-12.97,-6.814,-15.673,-16.548],"00418":["E",-14.469,-11.562,-14.547,-13.656,-0.0],"00419":["B",-17.829,-0.001,-7.657,-14.719,-18.86],"00420":["C",-5.945,-3.805,-0.039,-9.539,-4.32],"00421":["C",-7.488,-6.378,-0.003,-9.847,-6.956],"00422":["B",-0.725,-0.663,-9.6,-8.85,-14.1],"00423":["E",-4.635,-11.072,-13.869,-14.557,-0.01],"00425":["B",-1.858,-0.171,-7.53,-8.046,-12.233],"00426":["A",-0.0,-15.828,-15.531,-15.875,-17.609],"00428":["E",-1.265,-1.577,-8.515,-12.062,-0.671],"00429":["B",-8.031,-0.156,-1.937,-11.39,-14.031],"00433":["B",-4.904,-0.56,-1.154,-4.607,-2.342],"00436":["C",-18.719,-18.688,0.0,-17.344,-17.312],"00437":["E",-3.531,-2.813,-8.094,-9.828,-0.094],"00438":["B",-15.938,-0.0,-15.188,-17.188,-18.234],"00439":["D",-7.862,-6.096,-1.19,-0.44,-3.018],"00441":["A",-0.001,-7.344,-15.813,-18.063,-18.11],"00445":["A",-0.005,-8.505,-5.973,-12.958,-6.286],"00447":["E",-5.722,-9.535,-12.878,-10.785,-0.003],"00450":["C",-19.594,-17.625,0.0,-17.25,-18.656],"00451":["D",-11.039,-9.758,-12.368,-0.055,-2.93],"00457":["D",-15.656,-11.219,-13.469,-0.0,-15.547],"00460":["B",-14.438,-0.0,-12.906,-15.812,-15.812],"00465":["E",-6.47,-10.673,-14.72,-13.673,-0.002],"00467":["D",-1.696,-2.695,-8.664,-0.289,-16.961],"00471":["B",-8.688,-0.001,-11.501,-7.501,-12.173],"00473":["E",-12.906,-11.438,-15.797,-15.828,-0.0],"00474":["B",-20.297,-0.0,-16.25,-18.188,-18.438],"00475":["D",-5.023,-2.211,-2.461,-0.289,-3.008],"00476":["B",-15.781,-0.0,-13.125,-15.688,-17.234],"00479":["E",-12.891,-12.094,-12.469,-13.781,-0.0],"00480":["C",-9.282,-10.376,-0.001,-13.282,-7.282],"00481":["A",-0.0,-13.563,-13.813,-14.438,-8.375],"00483":["C",-6.439,-10.767,-0.002,-13.564,-11.314],"00484":["D",-9.969,-18.438,-18.25,-0.0,-19.375],"00485":["C",-9.545,-4.295,-0.233,-1.639,-14.28],"00487":["D",-5.909,-12.003,-10.284,-0.003,-12.659],"00488":["E",-21.266,-19.812,-17.312,-20.469,0.0],"00489":["B",-7.224,-0.005,-5.536,-8.63,-8.24],"00492":["B",-18.406,-0.0,-12.344,-17.906,-22.562],"00494":["C",-8.875,-11.906,-0.0,-10.516,-10.5],"00495":["B",-16.672,-0.0,-11.875,-13.656,-16.719],"00497":["B",-21.641,0.0,-19.25,-19.609,-21.203],"00499":["B",-10.286,-0.036,-3.349,-7.911,-10.005]},"longvideobench":{"-Xg2-SUq6wo_0":["A",-0.0,-12.563,-13.063,-10.219,-10.844],"-Xg2-SUq6wo_1":["D",-8.096,-7.189,-12.221,-0.002,-6.971],"-if_Wg43vTk_0":["D",-21.031,-20.594,-19.344,0.0,-17.719],"-kaF6SnSEo8_0":["B",-5.847,-0.003,-9.691,-15.847],"0WEnmqVVbHo_0":["B",-3.349,-0.412,-1.615,-3.818,-2.506],"0WEnmqVVbHo_1":["B",-10.658,-0.017,-6.548,-10.408,-4.173],"0ln2qdCR5lA_0":["B",-11.074,-0.012,-4.824,-12.512,-5.637],"0mEN5Jf2hU0_0":["C",-5.401,-4.542,-0.026,-4.573],"0mEN5Jf2hU0_1":["C",-4.595,-1.908,-0.439,-1.626],"0rWA-p4p5IM_1":["C",-3.597,-2.003,-0.254,-3.285,-3.722],"14ot4DrXdds_0":["D",-15.781,-8.406,-13.375,-0.0,-16.531],"1R5uPaL0V-0_1":["B",-8.625,-0.0,-12.0,-12.063,-13.969],"1oL4ugMdDvQ_0":["E",-13.674,-14.502,-6.252,-10.268,-0.002],"1pxrIj9Xyps_1":["D",-3.055,-4.149,-6.071,-0.149,-2.618],"2F7d7aUCmUU_0":["C",-3.401,-2.088,-0.182,-4.682],"2W2ZkYARds4_1":["D",-1.322,-5.932,-5.963,-0.385,-3.041],"2vVQo_GMA70_1":["E",-7.032,-9.642,-9.923,-9.454,-0.001],"2zZSMnGLGao_1":["D",-10.281,-17.344,-17.969,-0.0,-20.875],"3Qv2K_0GEE0_0":["B",-7.688,-0.001,-10.079,-12.11],"3avWNHoEDAg_0":["B",-19.656,-0.0,-12.094,-16.812,-20.438],"3hyPwjkdHEA_1":["D",-3.659,-3.128,-4.065,-0.143,-3.065],"3t_Knk7FWT8_0":["D",-22.219,-19.75,-19.375,0.0],"3u__SZlBLC0_0":["D",-2.01,-4.026,-0.916,-0.885,-3.338],"49YMA0f1yhI_0":["B",-20.5,0.0,-19.125,-19.562,-20.438],"4ouAf1ldH60_2":["E",-12.251,-10.907,-7.423,-9.141,-0.001],"58mCJVNMJvw_0":["A",-0.01,-8.729,-4.666,-9.557,-11.197],"5zbV24vyO44_1":["B",-6.861,-0.049,-3.439,-6.986,-4.314],"6d-EVupvWzU_1":["E",-4.383,-1.321,-2.571,-5.586,-0.446],"6moss7hGbvg_0":["B",-16.0,-0.0,-16.219,-14.094,-17.016],"7F9IrtSHmc0_1":["D",-12.408,-9.002,-2.127,-0.127],"7TljSpTBS9c_0":["A",-0.19,-2.659,-6.472,-5.784,-2.315],"88LbBgZP1vQ_1":["B",-2.469,-0.094,-6.938,-5.906,-6.688],"8MkL3W6wU3g_2":["B",-5.061,-0.671,-4.249,-0.764,-6.03],"8Qe03WDCrB4_0":["C",-13.562,-12.562,-0.0,-13.5,-13.688],"8hhcFRoR0mw_1":["A",-0.0,-15.562,-17.594,-16.5,-12.781],"8wbNEJjBa0k_0":["A",-0.703,-7.641,-3.703,-8.734,-0.734],"9PD3ciudpIE_1":["A",-0.417,-2.229,-3.26,-1.635],"9aWPbYosJUw_0":["D",-21.875,-18.844,-21.5,0.0],"9yQV0t5meo4_0":["D",-19.25,-15.969,-17.156,-0.0],"@_eat_sleep_travel_repeat-7296468146473536800_0":["B",-13.094,-0.0,-11.438,-13.609,-13.078],"@healthfood-6874943613630041350_0":["D",-25.078,-25.234,-22.75,0.0],"@healthfood-6948509102305791238_0":["B",-18.234,-0.0,-16.938,-16.375],"@healthfood-7115052132578921774_0":["C",-13.014,-13.03,-0.201,-13.201,-1.701],"@healthfood-7125042270381853995_0":["C",-20.234,-15.812,-0.0,-18.016],"@healthfood-7135100078221430058_0":["B",-11.188,-0.0,-13.0,-15.703],"@healthfood-7200404243168415022_0":["A",-0.031,-3.5,-10.625,-8.25,-11.422],"@jess.morg-7195624647830850858_0":["B",-14.317,-0.004,-5.504,-13.973,-15.41],"@jess.morg-7255843192677911851_0":["C",-14.984,-13.734,-0.0,-11.438,-13.094],"@jess.morg-7275475379375164715_0":["D",-14.406,-10.344,-14.188,-0.0,-16.781],"@jess.morg-7291800958152084782_0":["E",-12.797,-12.047,-12.359,-11.594,-0.0],"@jetset_anna-6933580627824495878_0":["A",-0.0,-12.844,-15.094,-16.438,-16.797],"@jetset_anna-7126584501228129542_0":["A",0.0,-20.844,-20.562,-23.062,-22.094],"@jonijawne-7184469162956246277_0":["E",-10.663,-5.069,-11.007,-3.476,-0.038],"@jonijawne-7197144530024500485_0":["D",-6.564,-10.58,-12.408,-0.002,-10.72],"@jonijawne-7214559428471770374_0":["A",-0.0,-17.875,-19.703,-15.719,-17.531],"@jonijawne-7305901654417706246_0":["B",-5.475,-0.006,-7.1,-7.694,-7.116],"@jonijawne-7312063059160206597_0":["D",-5.193,-14.943,-17.677,-0.006,-18.443],"@kelseyinlondon-7094322812327906565_0":["E",-6.476,-2.726,-7.867,-8.836,-0.07],"@kerstinong-6900436106790194434_0":["C",-3.886,-3.917,-0.261,-4.058,-1.761],"@kerstinong-7197781091162279169_0":["C",-20.266,-18.781,0.0,-20.594,-21.172],"@kerstinong-7254520467002854657_0":["D",-3.53,-12.78,-9.374,-0.03,-10.905],"@lisolna-7203500499360877829_0":["D",-13.984,-10.688,-9.594,-0.0,-16.188],"@lisolna-7235995901477604635_0":["E",-12.906,-9.125,-15.891,-16.485,-0.0],"@lisolna-7282789187676294432_0":["E",-15.032,-13.078,-13.578,-8.125,-0.0],"@lisolna-7288701821369978144_0":["D",-4.814,-1.939,-3.673,-0.204,-4.923],"@lisolna-7321010343067618592_0":["B",-10.791,-0.01,-7.01,-10.073,-4.713],"@lisolna-7359951777845775648_0":["A",-0.001,-8.704,-7.501,-8.704,-8.048],"@luxtravelbe-7233851486474636570_0":["B",-14.982,-0.013,-8.263,-6.607,-4.482],"@luxtravelbe-7286546546932370721_0":["A",-0.001,-11.626,-10.22,-6.907],"@movie.explained6-7253164232307379457_0":["B",-5.819,-0.006,-7.147,-6.069,-10.772],"@movie.explained6-7253792552996867330_0":["E",-5.816,-9.941,-10.659,-12.441,-0.003],"@movie.explained6-7254803496900267266_0":["A",-0.025,-12.103,-9.962,-12.697,-3.712],"@movie.explained6-7257359603489443073_0":["D",-19.062,-14.281,-17.875,-0.0,-19.938],"@movie.explained6-7259227637992705282_0":["A",-0.006,-12.974,-14.974,-13.381,-5.193],"@movie.explained6-7262938043315686664_0":["E",-4.054,-5.57,-7.085,-7.382,-0.023],"@movie.explained6-7268936523481943297_0":["A",-0.0,-13.625,-17.172,-15.875,-13.969],"@movie.explained6-7270293967017725185_0":["A",-0.957,-5.098,-1.051,-1.363,-5.426],"@movie.explained6-7275653401025826050_0":["B",-6.814,-0.001,-10.407,-11.689,-11.626],"@movie.explained6-7277140989259599105_0":["C",-3.304,-2.617,-0.117,-10.117,-8.523],"@movie.explained6-7277548218106367234_0":["D",-16.25,-15.719,-15.812,-0.0,-11.688],"@movie.explained6-7278310085217094914_0":["B",-5.548,-0.642,-3.079,-7.017,-0.861],"@placesunleashed-7289528053112343814_0":["B",-8.407,-0.313,-1.314,-15.532],"@placesunleashed-7297000148201262341_0":["A",0.0,-18.875,-19.906,-19.938,-18.844],"@placesunleashed-7303594391850044678_0":["D",-10.094,-10.906,-9.844,-0.0,-12.078],"@placesunleashed-7308098412078025989_0":["D",-10.485,-2.266,-7.594,-0.11],"@placesunleashed-7324498849857326341_0":["D",-21.469,-21.844,-17.781,0.0],"@placesunleashed-7337436458078260485_0":["C",-9.301,-6.005,-0.005,-6.223],"@recipesbyanne-7141686631676808454_0":["D",-9.815,-7.127,-6.596,-0.002,-9.643],"@recipesbyanne-7153617149041446150_0":["A",-0.033,-9.361,-4.877,-9.83,-3.69],"@recipesbyanne-7161798161395240197_0":["D",-16.25,-16.438,-16.812,-0.0,-17.125],"@recipesbyanne-7231566545263103259_0":["E",-8.719,-9.657,-10.125,-8.657,-0.0],"@thatrecipe.us-7266555380925287723_0":["D",-4.48,-11.183,-9.074,-0.012],"@thatrecipe.us-7301855783438896427_0":["D",-21.0,-19.906,-19.312,0.0],"@thatrecipe.us-7335998138500648235_0":["A",-0.53,-11.686,-11.998,-0.905,-5.03],"@tiffycooks-6940704678569053446_0":["C",-0.909,-2.816,-0.628,-5.597],"@tiffycooks-7062861483687873798_0":["B",-13.5,-0.0,-9.531,-14.625],"@tiffycooks-7080578381569445125_0":["C",-7.141,-1.766,-0.204,-4.376],"@tiffycooks-7171182579058511110_0":["B",-16.812,-0.0,-17.562,-14.281],"AHeq99pojLo_0":["D",-13.602,-11.868,-12.461,-0.305,-1.336],"Aau-XoIebno_2":["A",-0.015,-4.516,-8.437,-7.25,-5.656],"BtaVRhoLpC0_0":["B",-19.719,0.0,-20.188,-19.172],"CEZ9rbjK3P4_0":["A",-0.536,-11.88,-8.755,-0.88,-12.239],"DVsw1brd_Yc_0":["A",-0.0,-13.656,-15.422,-13.719],"DX-mYMYf8jI_0":["D",-12.75,-15.375,-16.438,-0.0],"Dkm35G5kkcc_0":["B",-1.288,-0.757,-1.663,-2.773,-5.726],"DoizYSYQRqU_2":["C",-2.709,-4.474,-0.693,-2.881,-1.006],"E7FSg22MdKE_1":["D",-11.11,-8.656,-10.125,-0.0,-9.781],"ErGYJ7kqIow_0":["C",-8.45,-7.2,-0.262,-6.075,-1.481],"F2OhCCEIOcU_0":["B",-3.229,-0.339,-1.464,-4.12],"F2OhCCEIOcU_1":["C",-1.176,-3.535,-0.91,-1.348],"F2i-LchFVCU_0":["C",-17.875,-15.969,-0.0,-15.594],"F4bDyyEO4PU_0":["A",-0.006,-10.225,-12.413,-14.975,-5.069],"F4bDyyEO4PU_1":["E",-12.58,-6.314,-14.408,-10.439,-0.002],"F8Ma1qs0Rkg_1":["B",-10.041,-0.01,-7.854,-4.666],"Fq3zbbp-lv4_1":["E",-2.833,-0.849,-3.396,-3.099,-0.833],"GAoK4XjssrM_0":["B",-11.344,-0.0,-13.406,-14.281],"GHScTyR2OgQ_0":["D",-4.91,-3.034,-9.503,-0.066,-4.816],"GZFL58_pXPg_0":["C",-6.534,-6.909,-0.002,-9.69],"GwcXYTX2GmA_0":["C",-1.761,-1.386,-0.73,-2.339],"H2ksp6sRR-k_2":["E",-2.829,-1.485,-6.313,-2.563,-0.454],"H7gnsO4UZmU_0":["A",-0.0,-12.406,-13.594,-14.219],"HAED3riiZkw_0":["D",-10.312,-11.75,-13.719,-0.0,-16.188],"HPunfsyjETs_0":["A",-0.548,-6.173,-0.954,-3.361],"HWyDOQrYtCk_0":["C",-11.328,-10.484,-0.0,-10.063],"I2Fyzav8MxE_0":["A",-0.0,-8.25,-9.672,-11.625,-11.438],"IN0osLg-Mn8_1":["E",-4.532,-4.61,-4.985,-5.485,-0.032],"Ip9DbdOtqF4_1":["D",-10.488,-9.8,-6.066,-0.003,-7.222],"J1IwKg2ufk8_2":["C",-16.391,-12.234,-0.0,-10.906,-16.531],"JIoE18JYGcM_0":["B",-20.5,-0.0,-16.469,-18.344,-15.094],"J_ZmaKRpyoU_2":["A",-0.47,-1.595,-2.329,-2.595],"Jfp1Ks7Hh1E_1":["E",-1.096,-7.393,-7.346,-8.175,-0.409],"Jfp1Ks7Hh1E_2":["A",-0.589,-2.464,-2.604,-1.37,-3.448],"Jim_NG0PYpc_0":["A",-0.026,-8.745,-9.651,-3.651],"JpjytHmGHZ4_0":["E",-22.094,-21.531,-25.109,-21.812,0.0],"LAQRjgx_OY8_1":["A",-0.46,-5.835,-1.023,-5.335,-6.491],"LYvStKy8iAc_1":["D",-11.563,-9.063,-13.031,-0.0,-18.485],"Lc7RikDaa30_0":["B",-1.799,-0.237,-4.971,-6.424,-3.299],"LcZxjqtzXJI_1":["E",-18.906,-18.938,-19.938,-18.969,0.0],"Lf7uW4xK2bc_0":["D",-22.469,-21.031,-21.438,0.0],"LfUsGv-ESbc_1":["D",-2.466,-2.576,-3.248,-0.248,-3.935],"LlqsCCa6y58_2":["A",-0.426,-2.692,-1.536,-5.098,-2.848],"M-YfPangEfA_0":["E",-12.641,-12.375,-13.719,-9.781,-0.0],"M5Umm-ltnao_0":["D",-1.246,-10.027,-11.574,-0.339],"M7YSCIkUaNw_1":["D",-10.688,-12.688,-11.406,-0.0,-10.828],"MJYBHfYF8LI_1":["B",-6.322,-0.041,-3.385,-5.541,-7.478],"MTmUo6QITo4_0":["C",-17.375,-15.375,-0.0,-10.594,-16.391],"Msz128EJeWE_0":["B",-11.517,-0.001,-7.063,-12.61],"MvgJAD6tZXo_2":["B",-3.144,-0.362,-1.893,-2.206,-9.112],"MyDAFlHsYsQ_0":["B",-14.562,-0.0,-16.172,-11.656,-15.062],"NHIT9vq6mJU_1":["A",-0.377,-1.158,-10.658,-12.877,-15.424],"Ng2rNm6Nwsg_0":["C",-2.762,-0.919,-0.669,-5.45,-3.856],"NpYUxd1vUUE_2":["A",-0.652,-4.215,-1.059,-6.73,-2.152],"O471uwTNx6k_2":["C",-1.627,-1.002,-0.97,-2.861],"P0BSTjziVys_0":["C",-7.188,-8.657,-0.001,-11.548],"PE32sjfN-uM_0":["B",-3.304,-0.086,-5.101,-3.242],"PbiTIR8N4Hc_1":["B",-1.756,-0.256,-3.022,-5.99,-6.225],"PzUxuZ-KGsU_0":["B",-16.5,-0.0,-13.062,-16.75],"QHS9ZZBdK-g_0":["C",-7.938,-8.094,-0.001,-12.219],"QPth_xqBXGY_2":["E",-2.425,-2.862,-5.487,-4.55,-0.175],"RTUFPjliMCU_0":["A",-0.481,-3.34,-1.512,-3.84,-2.262],"SYqEMs0EYoI_0":["A",-0.098,-4.614,-3.051,-3.317],"Sn7JPKbG6tY_2":["A",-0.008,-6.633,-8.071,-5.321,-6.352],"T15Kv6dtYO0_0":["C",-17.266,-16.109,-0.0,-15.078,-16.391],"T15Kv6dtYO0_2":["A",-0.001,-10.782,-8.5,-13.297,-8.125],"TMe7oXMJoSM_2":["C",-8.789,-7.492,-0.007,-5.007,-11.945],"TXiX3NO5f5w_0":["B",-10.813,-0.0,-8.406,-9.969],"TlaX2iIYZD4_1":["C",-14.781,-11.688,-0.0,-10.922,-8.906],"TynnVVRNZGs_0":["D",-12.281,-12.438,-11.938,-0.0],"UD5ifzOPzhc_0":["E",-9.853,-10.978,-9.759,-3.712,-0.025],"UN3ICsfqKEY_1":["E",-10.41,-5.691,-12.222,-12.425,-0.003],"UbNyMSwoT5A_0":["B",-1.931,-0.9,-2.681,-3.15,-1.087],"UbgwG8fcIu0_0":["A",-0.0,-9.156,-8.625,-12.156],"UwJTCg5fpXg_1":["C",-4.859,-3.297,-0.047,-9.766,-6.797],"Uy_o-WCq2Cc_0":["A",-0.0,-13.156,-15.969,-18.5,-17.656],"V0h7rJShw0g_0":["C",-12.141,-8.469,-0.0,-10.094],"VLInjyogciw_0":["E",-10.836,-0.758,-10.664,-9.039,-0.633],"VNvH5u5CWV8_0":["B",-3.255,-0.068,-4.052,-4.693,-7.99],"VPcdtoldWCA_0":["A",-0.0,-15.094,-16.281,-16.344,-17.719],"VkNF0rXuDXw_1":["B",-13.375,-0.0,-14.031,-14.062,-12.0],"Vn-z0qXp9bQ_0":["A",-0.001,-9.095,-6.689,-11.095],"VwZeSoYugZk_1":["C",-14.469,-11.781,-0.0,-10.719,-9.375],"WHxBO7XecSY_0":["E",-15.845,-17.251,-17.189,-6.689,-0.001],"WaiGdRYD36k_1":["A",-0.0,-9.594,-8.094,-11.985],"X5v4nBo5y28_2":["D",-2.084,-2.443,-1.756,-0.553,-3.209],"XJ6REZOXsvM_0":["E",-7.665,-11.227,-4.852,-10.649,-0.008],"XVXczyheik0_1":["B",-10.907,-0.001,-6.97,-8.501,-12.361],"Y1YCvEip_ko_2":["D",-13.25,-14.375,-15.328,-0.0,-15.141],"YcbKamVxDzI_1":["D",-4.175,-1.722,-3.347,-0.878,-1.035],"YcbKamVxDzI_2":["E",-2.033,-1.314,-2.595,-2.923,-0.751],"YehqA9xoGTY_1":["B",-12.196,-0.055,-11.024,-3.805,-3.461],"YiHjsOqxtk0_0":["D",-12.533,-12.267,-6.533,-0.002,-11.033],"Z6Hx_BAZeUw_0":["E",-12.094,-11.062,-13.516,-13.625,-0.0],"Z6Hx_BAZeUw_2":["E",-10.703,-14.313,-9.094,-8.438,-0.0],"Z7Cox6lPW3c_1":["B",-20.359,0.0,-16.938,-17.469,-19.219],"ZoUsR8t8IxE_0":["D",-6.497,-1.685,-4.904,-0.216],"ZoUsR8t8IxE_1":["D",-11.969,-9.875,-12.031,-0.0],"_EUDpS9UF9o_0":["A",-0.0,-9.656,-13.719,-12.219],"aVHAr8rc-Ks_1":["B",-12.234,-0.0,-11.906,-10.234,-9.516],"aX_HgA5SNLQ_0":["A",-0.789,-0.805,-2.82,-3.274,-6.68],"akoJDx23QWU_0":["E",-8.282,-9.438,-11.907,-11.407,-0.0],"anQ5KFW8Gn4_0":["D",-0.994,-12.369,-12.712,-0.462],"aqUisZS9Ruw_0":["A",-1.049,-1.268,-1.455,-2.94,-2.502],"aqUisZS9Ruw_1":["D",-1.418,-2.183,-3.074,-0.558,-3.621],"bDpgz_2Piqg_0":["C",-12.053,-2.521,-0.084,-8.834],"bH1Q9AM5vik_0":["A",-0.005,-10.817,-11.567,-5.38],"baFsMWNavQ4_0":["B",-1.809,-1.122,-2.7,-1.153,-2.059],"d8H7hgQY9ew_0":["C",-13.219,-14.156,-0.0,-13.125,-13.312],"d8H7hgQY9ew_1":["C",-6.318,-6.521,-0.006,-8.568,-6.131],"dE5iWeCVpGI_0":["B",-3.551,-0.458,-1.504,-2.614,-3.145],"e-EtQHH4aWE_0":["A",-0.235,-1.579,-5.782,-8.079],"e11Q4ThFu5A_0":["B",-6.252,-0.002,-11.361,-11.643,-12.486],"eE5Z7gDbgVA_1":["C",-8.673,-4.485,-0.017,-7.345,-5.392],"e_nAKiSutiA_0":["A",-0.0,-12.344,-12.938,-10.219],"er1oRjH2iu8_1":["B",-4.028,-0.122,-3.559,-2.684],"fO7nwCix8xU_0":["E",-1.132,-4.788,-5.07,-6.445,-0.413],"fPLjjr8w6DU_1":["A",-0.157,-2.064,-4.267,-5.642,-6.829],"fZBC3nmvJb8_0":["D",-10.041,-8.212,-1.478,-0.259,-10.009],"fryyNwUCPWA_1":["B",-5.347,-0.144,-2.05,-8.394,-9.941],"fvCrE5NCsts_1":["C",-5.787,-7.443,-0.006,-9.818,-6.193],"g7zuBUMBr2E_0":["E",-6.456,-6.018,-3.55,-7.237,-0.034],"gAgCnu82RHE_1":["D",-9.42,-4.623,-2.248,-0.123,-10.357],"gJuOMPiixUA_0":["A",-0.0,-13.344,-16.719,-13.531],"gtX_oRpLClY_0":["C",-2.115,-0.99,-0.897,-2.334,-5.74],"gyV6EqgiPNg_0":["D",-17.297,-15.0,-7.907,-0.0],"h0OHi9uAcBo_0":["D",-7.046,-4.327,-8.577,-0.014],"h4jIoMxZopU_0":["E",-21.188,-19.188,-18.875,-20.656,0.0],"h4jIoMxZopU_1":["A",-0.407,-3.595,-1.188,-8.032,-6.235],"hf4WUOagFAw_1":["D",-1.077,-3.155,-2.467,-0.999,-1.811],"hznvV2bBkX4_0":["D",-4.71,-4.179,-0.96,-0.523],"i6C6r2g4Y7Q_0":["B",-5.608,-0.264,-5.577,-6.561,-1.499],"i8TJ7RgimNM_0":["D",-26.062,-25.375,-24.562,0.0],"iHNjWhx3EaI_0":["D",-13.518,-13.706,-5.909,-0.003,-12.472],"iJgh2dnudIU_1":["B",-8.188,-0.001,-7.97,-9.094],"ie9NxdgNe0g_0":["C",-15.422,-14.391,-0.0,-15.703],"iwXp1fT89-M_1":["A",-0.547,-4.656,-1.391,-2.266,-2.828],"iwXp1fT89-M_2":["A",-0.137,-4.637,-3.7,-2.418,-5.34],"ixAU3l0sX_o_0":["D",-18.469,-15.812,-14.781,-0.0],"jJGbXCCU5yc_0":["B",-6.803,-0.303,-2.256,-3.053,-2.225],"jbPR2SJuFHg_0":["C",-11.346,-6.221,-0.002,-10.096],"jvkmcX47bKU_1":["C",-12.203,-10.375,-0.0,-13.219,-10.594],"jvkmcX47bKU_2":["B",-3.563,-0.094,-5.469,-5.5,-2.938],"k194yU67Zik_0":["C",-3.09,-1.809,-0.669,-1.934,-2.012],"k4jiEuZbN-4_0":["E",-1.573,-3.198,-1.635,-3.604,-0.636],"kFHVBCEwC3w_0":["E",-6.189,-6.345,-3.033,-4.611,-0.064],"kWUmHAzCp7s_0":["C",-11.659,-6.659,-0.002,-10.455,-6.877],"kbRtl58u_kk_0":["D",-25.781,-22.938,-23.719,0.0],"l3vxOGgAM2g_0":["D",-10.269,-10.16,-2.379,-0.097],"lN3WnXMaE0o_0":["E",-3.929,-3.945,-1.711,-1.539,-0.57],"lN3WnXMaE0o_1":["E",-2.444,-3.382,-3.616,-6.006,-0.163],"lNReCCShKJQ_0":["C",-7.456,-4.8,-0.019,-4.644],"lcEaHk8f4Co_0":["B",-6.19,-0.002,-9.924,-11.237,-7.815],"luRqMb5qfhM_0":["B",-14.5,-0.0,-15.844,-14.594],"m1mGerQhdHA_0":["A",-0.0,-15.25,-16.062,-10.844],"m7sL8LuMD8Q_0":["C",-1.052,-1.802,-0.74,-4.74],"mB7NW91REF4_0":["A",-0.042,-3.386,-4.917,-9.62],"mFcEWmtn3ag_1":["B",-5.075,-0.669,-7.95,-0.732],"mHARxee4EzQ_0":["A",-0.0,-12.016,-13.953,-14.203,-9.625],"mS1QPVgBDQo_1":["D",-13.75,-11.203,-8.063,-0.0,-13.5],"mS1QPVgBDQo_2":["B",-1.293,-0.325,-7.231,-6.543,-7.168],"mTn_C-SyW84_0":["E",-5.411,-7.395,-5.442,-4.551,-0.02],"mbcvVYobCXI_1":["E",-4.216,-2.591,-4.888,-1.778,-0.31],"mld0TnA2jEs_1":["C",-14.375,-12.406,-0.0,-14.047,-13.656],"nBWB-h68E4k_0":["D",-2.139,-1.764,-7.248,-0.342],"nvG2YMWiwqU_0":["B",-17.75,-0.0,-13.156,-16.906],"o2F-N42Ufo4_2":["D",-0.895,-3.442,-3.583,-0.677,-3.755],"oCXKARwr6PA_0":["C",-11.457,-10.598,-0.004,-5.566,-10.426],"oO8OJZFZT9s_0":["A",-0.525,-1.306,-3.275,-2.306],"oddHY1vwcjo_0":["A",-0.632,-5.336,-1.101,-2.039,-6.633],"pFtKaT3GF9I_0":["A",-1.054,-1.054,-1.21,-5.444,-8.319],"pPJq1rMDRGs_1":["A",-0.651,-2.058,-2.464,-4.433,-1.37],"q3FAxTSENEw_1":["A",-0.644,-4.66,-2.269,-1.019,-6.863],"qZVBFAtfp2A_0":["D",-9.149,-4.149,-9.243,-0.024,-4.899],"qbA42wQoWAs_0":["B",-4.047,-0.454,-5.032,-7.688,-1.079],"qksR2Zvd-FM_2":["C",-2.338,-2.213,-0.729,-1.166],"qsH6q5wNso4_1":["C",-1.816,-1.128,-1.035,-1.847,-7.191],"rP7sQe784k8_1":["E",-3.623,-7.108,-7.701,-6.951,-0.029],"rwL_XPw46zQ_1":["A",-0.027,-5.589,-4.902,-4.214,-7.214],"rzIiQ4Vxlbk_0":["E",-8.305,-2.93,-9.915,-12.493,-0.055],"t1nhAnMQBHg_0":["E",-10.185,-1.013,-10.435,-10.232,-0.451],"t1nhAnMQBHg_1":["B",-17.703,-0.0,-15.344,-16.594,-16.75],"t3P11ENBZyc_1":["C",-9.648,-11.57,-0.523,-10.539,-0.898],"tdA5atpqaAc_0":["B",-12.134,-0.009,-13.712,-4.759,-16.212],"u1D4ArcBjLI_1":["D",-1.582,-6.519,-5.394,-0.238,-10.8],"u5NAcHhI_Uc_0":["E",-20.781,-20.875,-21.812,-19.594,0.0],"uDZrl35yeZE_0":["D",-10.226,-12.319,-14.148,-0.007,-4.976],"v-7zF3Y0yJs_0":["D",-5.473,-14.801,-12.004,-0.004,-16.785],"v0QLje6xYgA_0":["C",-3.496,-2.621,-0.543,-1.199,-4.261],"vJ9hYCUDHTo_0":["B",-2.401,-0.588,-1.057,-7.963,-5.088],"vJ9hYCUDHTo_1":["C",-4.533,-3.487,-0.08,-5.268,-3.487],"vVRC-0VKPrg_2":["E",-12.828,-12.375,-11.469,-10.453,-0.0],"vix2_R79-l4_0":["B",-10.658,-0.002,-6.533,-8.845],"wGWbI3BHVM8_0":["A",-0.258,-7.726,-5.258,-1.508],"wSHPuI7wWIg_2":["C",-1.777,-3.214,-0.636,-1.527,-3.121],"xSzXFQPldJg_0":["C",-2.55,-9.863,-0.081,-13.16],"yFAuXmcGk2Y_0":["E",-0.838,-6.072,-3.197,-6.135,-0.65],"yqejTvYILlA_2":["A",-0.073,-2.73,-6.448,-5.964,-6.636],"ysRFFN5nzqE_2":["A",-0.0,-10.656,-8.688,-14.078,-12.922],"yz3lOAe32Tw_0":["E",-3.399,-1.711,-5.18,-3.93,-0.274],"z8YF-D-t7AI_0":["B",-21.062,0.0,-20.75,-22.031,-22.188],"zVudr8cxHRE_1":["D",-6.164,-4.336,-1.742,-0.211]},"videomme":{"165-1":["C",-4.674,-1.455,-0.283,-5.549],"165-2":["C",-6.088,-4.948,-0.01,-7.01],"165-3":["A",0.0,-18.25,-19.594,-20.812],"180-1":["D",-8.503,-5.816,-11.175,-0.003],"180-2":["B",-7.359,-0.046,-3.109,-11.64],"180-3":["C",-8.0,-10.875,-0.0,-13.985],"182-1":["C",-14.188,-14.0,-0.0,-15.594],"182-2":["B",-3.533,-0.033,-5.908,-7.705],"182-3":["D",-9.688,-13.875,-13.703,-0.0],"206-1":["B",-2.018,-0.143,-9.8,-7.596],"206-2":["B",-3.445,-0.524,-1.023,-4.117],"206-3":["B",-4.11,-0.017,-8.423,-10.907],"240-1":["A",-0.0,-16.312,-14.719,-17.281],"240-2":["A",-0.004,-9.286,-12.676,-5.442],"240-3":["B",-5.691,-0.004,-10.441,-7.629],"260-1":["D",-17.406,-12.719,-13.312,-0.0],"260-2":["D",-3.368,-4.634,-4.603,-0.056],"260-3":["C",-10.235,-9.454,-0.001,-7.298],"266-1":["D",-17.188,-13.359,-12.906,-0.0],"266-2":["C",-18.672,-17.188,0.0,-19.344],"266-3":["A",-0.077,-2.608,-8.608,-10.467],"298-1":["C",-1.025,-2.149,-0.915,-2.087],"298-2":["A",-0.4,-1.462,-2.462,-4.384],"298-3":["C",-2.756,-2.553,-0.272,-2.335],"329-1":["B",-8.156,-0.0,-12.828,-16.156],"329-2":["B",-15.375,-0.0,-14.562,-14.844],"329-3":["A",-0.0,-9.906,-14.328,-13.969],"334-1":["C",-9.688,-12.781,-0.0,-18.406],"334-2":["B",-6.306,-0.04,-3.29,-10.821],"334-3":["C",-3.323,-2.589,-0.12,-6.105],"380-1":["D",-13.219,-11.625,-11.656,-0.0],"380-2":["B",-1.608,-1.108,-1.608,-1.311],"380-3":["C",-21.203,-18.922,-0.0,-16.156],"402-1":["A",-0.622,-2.184,-1.059,-5.637],"402-2":["D",-3.778,-7.263,-5.435,-0.028],"402-3":["B",-6.785,-0.003,-6.128,-11.644],"419-1":["B",-14.438,-0.0,-18.344,-18.375],"419-2":["D",-9.934,-8.871,-4.48,-0.012],"419-3":["B",-12.427,-0.005,-5.318,-8.849],"428-1":["A",-0.001,-10.251,-6.595,-13.673],"428-2":["B",-16.188,-0.0,-16.562,-16.812],"428-3":["C",-8.329,-8.439,-0.001,-7.532],"447-1":["D",-8.313,-16.203,-12.281,-0.0],"447-2":["B",-1.657,-0.704,-1.719,-2.0],"447-3":["A",-1.089,-4.121,-1.152,-1.105],"456-1":["B",-13.313,-0.0,-12.641,-7.719],"456-2":["B",-2.462,-0.587,-1.04,-5.259],"456-3":["B",-3.028,-0.637,-0.981,-3.043],"586-1":["B",-1.576,-0.733,-1.92,-1.795],"586-2":["A",-0.001,-7.095,-17.032,-16.11],"586-3":["D",-3.98,-3.386,-5.605,-0.058],"722-1":["C",-7.714,-6.776,-0.01,-4.745],"722-2":["A",-0.451,-11.92,-13.373,-1.013],"722-3":["A",-0.032,-4.47,-4.517,-4.657],"724-1":["A",-0.0,-12.031,-12.891,-12.516],"724-2":["D",-6.063,-0.813,-6.313,-0.594],"724-3":["A",-0.887,-1.778,-1.293,-1.934],"744-1":["C",-2.38,-2.88,-0.162,-7.724],"744-2":["B",-6.978,-0.009,-6.165,-5.149],"744-3":["B",-2.024,-0.446,-1.508,-5.055],"747-1":["C",-5.079,-1.658,-0.298,-2.798],"747-2":["B",-1.463,-0.744,-1.416,-2.979],"747-3":["D",-2.628,-2.221,-1.284,-0.612],"779-1":["A",-1.162,-1.178,-1.318,-2.193],"779-2":["C",-11.422,-10.891,-0.0,-12.781],"779-3":["A",-0.0,-13.609,-9.5,-16.031],"856-1":["C",-6.816,-2.269,-0.113,-6.05],"856-2":["D",-4.914,-7.024,-3.493,-0.039],"856-3":["C",-7.532,-8.61,-0.001,-8.095],"875-1":["B",-14.812,-0.0,-18.422,-17.516],"875-2":["B",-7.396,-0.005,-10.364,-5.443],"875-3":["C",-11.078,-10.781,-0.0,-9.969],"038-1":["A",-0.05,-6.628,-4.581,-3.3],"038-2":["A",-0.019,-15.097,-4.113,-6.081],"038-3":["A",-0.0,-14.938,-15.344,-11.188],"219-1":["B",-19.875,0.0,-21.0,-19.5],"219-2":["C",-7.1,-4.303,-0.647,-0.772],"219-3":["B",-8.032,-0.001,-8.282,-12.079],"224-1":["C",-15.875,-17.828,-0.0,-16.5],"224-2":["A",-0.002,-6.377,-9.799,-12.142],"224-3":["B",-10.202,-0.015,-11.765,-4.233],"246-1":["B",-21.25,0.0,-23.375,-22.125],"246-2":["B",-4.918,-0.152,-2.668,-2.746],"246-3":["C",-2.771,-7.411,-0.068,-6.13],"270-1":["C",-11.797,-10.641,-0.0,-13.453],"270-2":["C",-13.109,-9.844,-0.0,-10.688],"270-3":["B",-18.219,0.0,-17.25,-18.125],"335-1":["A",-0.174,-6.581,-3.659,-2.018],"335-2":["D",-19.781,-18.688,-19.438,0.0],"335-3":["A",-0.0,-15.594,-15.281,-15.625],"344-1":["C",-12.016,-9.094,-0.0,-11.656],"344-2":["D",-6.273,-1.367,-5.133,-0.305],"344-3":["A",-1.116,-1.116,-1.116,-4.054],"361-1":["A",-0.0,-14.531,-17.938,-14.625],"361-2":["D",-7.605,-6.777,-2.668,-0.074],"361-3":["B",-17.938,-0.0,-9.094,-17.875],"433-1":["D",-3.6,-1.865,-5.022,-0.209],"433-2":["B",-7.192,-0.082,-2.551,-9.691],"433-3":["C",-7.919,-4.669,-0.184,-1.841],"437-1":["C",-12.301,-5.566,-0.004,-12.144],"437-2":["C",-19.031,-15.969,-0.0,-19.078],"437-3":["A",-0.008,-6.961,-5.102,-6.961],"527-1":["C",-1.105,-1.949,-0.824,-2.433],"527-2":["C",-6.076,-3.639,-0.03,-7.67],"527-3":["B",-19.656,-0.0,-16.062,-18.031],"623-1":["D",-12.593,-12.609,-1.937,-0.156],"623-2":["D",-4.584,-5.146,-5.318,-0.021],"623-3":["D",-16.688,-17.375,-18.328,-0.0],"674-1":["C",-14.859,-12.312,-0.0,-17.156],"674-2":["B",-3.168,-0.043,-9.778,-7.762],"674-3":["B",-1.462,-0.884,-1.166,-3.134],"762-1":["C",-6.752,-2.408,-0.096,-7.721],"762-2":["B",-7.198,-0.635,-0.807,-3.76],"762-3":["C",-7.215,-2.543,-0.105,-3.886],"768-1":["C",-8.402,-0.917,-0.511,-7.73],"768-2":["A",-0.01,-5.651,-5.104,-7.151],"768-3":["A",-0.006,-7.319,-8.522,-5.256],"774-1":["C",-21.547,-22.172,0.0,-19.688],"774-2":["B",-5.96,-0.117,-3.335,-2.632],"774-3":["A",-0.011,-8.542,-4.823,-5.98],"784-1":["D",-19.469,-19.672,-18.188,0.0],"784-2":["C",-8.704,-7.298,-0.001,-9.438],"784-3":["B",-11.109,-0.0,-13.063,-9.313],"014-1":["D",-9.813,-12.547,-12.547,-0.0],"014-2":["D",-19.234,-18.016,-14.062,-0.0],"014-3":["B",-6.145,-0.004,-6.879,-6.614],"075-1":["B",-16.25,-0.0,-14.375,-15.469],"075-2":["C",-11.938,-11.812,-0.0,-13.156],"075-3":["A",0.0,-18.719,-22.281,-20.875],"133-1":["B",-0.913,-0.616,-3.116,-4.241],"133-2":["B",-1.239,-0.645,-1.708,-5.426],"133-3":["D",-5.069,-10.475,-9.1,-0.006],"228-1":["C",-20.688,-20.438,0.0,-20.219],"228-2":["A",-0.119,-2.603,-3.338,-5.9],"228-3":["C",-0.758,-9.867,-0.633,-10.414],"283-1":["A",-0.066,-2.785,-7.082,-6.613],"283-2":["D",-21.156,-19.5,-18.312,0.0],"283-3":["D",-5.514,-0.999,-7.014,-0.467],"323-1":["A",-0.0,-11.063,-10.156,-12.25],"323-2":["B",-8.844,-0.0,-9.469,-10.594],"323-3":["C",-9.165,-1.18,-0.368,-7.477],"349-1":["C",-20.5,-20.906,0.0,-22.234],"349-2":["A",-0.0,-9.813,-9.906,-10.922],"349-3":["A",-0.0,-9.531,-12.125,-11.625],"362-1":["C",-18.312,-20.25,0.0,-19.859],"362-2":["A",-0.799,-2.721,-2.596,-0.892],"362-3":["D",-10.469,-12.922,-13.359,-0.0],"390-1":["B",-1.219,-0.391,-3.609,-7.109],"390-2":["C",-9.142,-7.282,-0.001,-7.939],"390-3":["B",-16.25,-0.0,-17.594,-16.594],"394-1":["D",-16.625,-15.312,-15.125,-0.0],"394-2":["B",-3.745,-0.401,-1.182,-9.776],"394-3":["D",-3.984,-2.39,-1.296,-0.484],"410-1":["B",-6.726,-0.07,-2.726,-7.445],"410-2":["A",-0.001,-7.626,-11.782,-13.672],"410-3":["B",-4.594,-0.031,-3.922,-7.094],"429-1":["D",-1.846,-2.44,-6.159,-0.284],"429-2":["D",-13.812,-16.547,-14.438,-0.0],"429-3":["D",-3.48,-2.292,-0.933,-0.745],"449-1":["A",-0.34,-1.387,-4.293,-3.699],"449-2":["B",-1.123,-0.42,-4.077,-6.811],"449-3":["B",-4.124,-0.483,-3.171,-1.124],"461-1":["A",-0.0,-12.031,-12.344,-15.188],"461-2":["C",-6.465,-4.606,-0.012,-8.09],"461-3":["B",-1.204,-0.486,-2.564,-4.845],"463-1":["C",-2.051,-1.004,-0.691,-5.504],"463-2":["B",-5.05,-0.643,-0.768,-5.456],"463-3":["A",-0.003,-5.785,-9.16,-8.832],"466-1":["A",-0.155,-4.562,-2.015,-9.14],"466-2":["C",-1.387,-5.996,-0.293,-6.371],"466-3":["B",-1.036,-0.442,-9.942,-6.176],"499-1":["C",-2.83,-2.049,-0.517,-1.533],"499-2":["C",-18.156,-16.625,-0.0,-18.266],"499-3":["A",-0.731,-0.731,-3.528,-4.872],"507-1":["A",0.0,-20.75,-20.562,-19.812],"507-2":["A",-0.004,-6.286,-6.317,-7.145],"507-3":["D",-9.928,-0.975,-6.803,-0.475],"538-1":["C",-5.569,-6.1,-0.006,-10.553],"538-2":["B",-5.065,-0.221,-2.174,-2.549],"538-3":["B",-2.94,-0.127,-2.721,-7.283],"606-1":["A",-0.465,-2.574,-1.496,-2.637],"606-2":["C",-4.884,-4.087,-0.025,-9.306],"606-3":["B",-2.843,-0.062,-7.781,-6.578],"726-1":["A",-0.004,-9.067,-8.192,-5.504],"726-2":["B",-1.682,-0.917,-1.151,-2.323],"726-3":["D",-7.148,-1.632,-5.163,-0.226],"761-1":["B",-2.114,-0.473,-1.457,-3.77],"761-2":["C",-6.97,-11.22,-0.001,-8.126],"761-3":["C",-4.804,-6.476,-0.023,-4.336],"802-1":["D",-7.749,-8.358,-0.936,-0.499],"802-2":["A",-0.185,-2.513,-2.779,-3.654],"802-3":["C",-2.261,-2.214,-0.823,-1.058],"838-1":["C",-4.88,-4.38,-0.021,-8.052],"838-2":["B",-3.5,-0.032,-7.36,-7.719],"838-3":["B",-16.188,-0.0,-13.875,-14.219],"859-1":["C",-2.367,-1.07,-0.867,-1.945],"859-2":["D",-7.417,-4.964,-5.901,-0.01],"859-3":["C",-1.711,-2.883,-0.305,-3.648],"862-1":["A",-0.007,-10.507,-5.257,-6.304],"862-2":["A",-0.975,-4.85,-1.303,-1.069],"862-3":["C",-4.575,-6.293,-0.012,-12.168],"894-1":["D",-4.494,-2.041,-1.354,-0.51],"894-2":["D",-12.516,-7.938,-9.657,-0.0],"894-3":["A",-0.524,-1.431,-2.259,-2.743],"026-1":["C",-24.391,-22.406,0.0,-22.047],"026-2":["A",-0.0,-15.156,-15.562,-17.219],"026-3":["D",-9.844,-14.563,-12.703,-0.0],"032-1":["B",-9.531,-0.0,-12.375,-14.75],"032-2":["C",-1.895,-3.114,-0.239,-4.036],"032-3":["B",-2.616,-0.538,-2.006,-1.569],"066-1":["D",-25.578,-24.438,-24.125,0.0],"066-2":["B",-11.688,-0.0,-19.016,-16.062],"066-3":["C",-14.203,-10.5,-0.0,-13.141],"126-1":["D",-16.0,-12.906,-9.25,-0.0],"126-2":["B",-14.844,-0.0,-16.594,-17.859],"126-3":["B",-18.984,-0.0,-15.531,-17.094],"188-1":["A",-0.434,-1.184,-3.075,-8.856],"188-2":["A",-0.0,-15.094,-18.25,-16.0],"188-3":["B",-19.844,0.0,-18.938,-19.5],"210-1":["A",-0.589,-3.73,-0.87,-6.027],"210-2":["B",-21.516,0.0,-19.562,-19.25],"210-3":["C",-0.884,-2.884,-0.634,-9.134],"352-1":["B",-16.0,-0.0,-16.406,-15.562],"352-2":["D",-11.735,-8.125,-10.891,-0.0],"352-3":["A",-0.215,-1.84,-3.543,-5.106],"370-1":["C",-12.781,-9.406,-0.0,-15.906],"370-2":["A",-0.004,-5.442,-10.629,-12.036],"370-3":["C",-4.482,-3.919,-0.06,-3.607],"412-1":["A",-0.269,-7.863,-1.488,-4.66],"412-2":["B",-6.003,-0.003,-7.628,-8.16],"412-3":["C",-13.125,-14.156,-0.0,-14.297],"452-1":["C",-17.125,-15.484,-0.0,-16.172],"452-2":["C",-3.581,-1.675,-0.41,-2.113],"452-3":["C",-2.883,-1.102,-0.696,-2.18],"502-1":["D",-22.062,-21.656,-22.156,0.0],"502-2":["C",-3.68,-1.118,-0.446,-4.915],"502-3":["C",-1.887,-1.871,-0.418,-3.324],"506-1":["D",-3.138,-8.404,-14.623,-0.045],"506-2":["D",-3.103,-3.431,-3.025,-0.134],"506-3":["D",-4.392,-9.876,-5.439,-0.017],"547-1":["C",-4.848,-1.598,-0.238,-6.238],"547-2":["A",-0.147,-2.522,-2.897,-7.037],"547-3":["A",-0.689,-1.814,-1.298,-2.783],"556-1":["B",-9.125,-0.001,-9.25,-8.141],"556-2":["C",-1.36,-7.126,-0.298,-9.579],"556-3":["D",-10.173,-6.814,-7.923,-0.002],"680-1":["D",-3.469,-2.813,-3.204,-0.141],"680-2":["B",-11.334,-0.006,-9.709,-5.193],"680-3":["C",-8.969,-11.985,-0.0,-9.656],"705-1":["C",-1.938,-3.094,-0.469,-1.688],"705-2":["D",-11.126,-11.501,-7.126,-0.001],"705-3":["C",-2.61,-6.766,-0.078,-7.36],"709-1":["C",-6.599,-0.864,-0.552,-6.63],"709-2":["C",-3.662,-2.599,-0.115,-4.787],"709-3":["C",-4.496,-0.996,-0.543,-3.261],"713-1":["B",-6.51,-0.119,-2.525,-3.479],"713-2":["C",-3.349,-2.349,-0.614,-1.114],"713-3":["D",-5.839,-2.511,-3.527,-0.12],"794-1":["C",-16.141,-10.031,-0.0,-16.312],"794-2":["B",-5.566,-0.004,-10.582,-10.691],"794-3":["A",-0.059,-8.106,-2.871,-8.059],"840-1":["B",-2.562,-0.296,-1.952,-3.296],"840-2":["B",-3.103,-0.932,-1.369,-1.181],"840-3":["C",-1.292,-2.464,-0.932,-1.401],"163-1":["A",-0.171,-1.936,-4.39,-7.765],"163-2":["D",-24.578,-20.156,-22.062,0.0],"163-3":["B",-2.403,-0.637,-1.137,-2.809],"258-1":["C",-4.058,-4.652,-0.027,-9.246],"258-2":["A",-0.0,-10.156,-12.594,-11.766],"258-3":["D",-18.812,-19.812,-21.031,0.0],"269-1":["A",-0.0,-15.375,-17.438,-16.094],"269-2":["D",-10.844,-10.563,-8.688,-0.0],"269-3":["D",-17.484,-11.969,-13.938,-0.0],"275-1":["A",-0.0,-16.062,-19.844,-20.844],"275-2":["C",-19.672,-18.344,0.0,-18.734],"275-3":["D",-20.219,-18.875,-22.656,0.0],"332-1":["C",-10.938,-9.735,-0.0,-7.922],"332-2":["D",-1.839,-1.464,-1.901,-0.776],"332-3":["A",-0.231,-1.637,-5.106,-5.168],"355-1":["C",-13.151,-11.87,-0.01,-4.573],"355-2":["B",-10.828,-0.0,-9.578,-8.797],"355-3":["C",-18.375,-17.766,0.0,-17.156],"465-1":["C",-7.726,-5.039,-0.007,-7.789],"465-2":["C",-4.613,-2.675,-0.097,-4.269],"465-3":["D",-2.541,-4.073,-5.854,-0.104],"534-1":["A",-0.022,-13.053,-11.772,-3.834],"534-2":["B",-5.075,-0.012,-5.231,-7.325],"534-3":["C",-8.142,-7.47,-0.001,-9.985],"555-1":["A",-0.0,-13.734,-11.75,-10.375],"555-2":["C",-3.581,-2.924,-0.424,-1.331],"555-3":["D",-5.89,-0.875,-4.578,-0.562],"867-1":["D",-4.52,-3.458,-2.536,-0.13],"867-2":["C",-5.57,-4.539,-0.164,-1.992],"867-3":["B",-1.743,-0.509,-2.571,-1.915],"871-1":["D",-19.219,-18.75,-18.844,0.0],"871-2":["B",-9.246,-0.199,-1.731,-5.621],"871-3":["B",-14.859,-0.0,-14.625,-11.156],"879-1":["D",-4.905,-2.733,-1.577,-0.327],"879-2":["C",-10.156,-9.844,-0.0,-9.828],"879-3":["D",-11.75,-18.141,-16.062,-0.0]}}
print({b: len(v) for b, v in REFERENCE_FULL.items()}, 'reference answers')

### 1 · Setup

In [ ]:
# ── Setup: folders, environment, small installs ──────────────────────
import os, sys, subprocess
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(WORK_DIR, "hf_home")      # keep HF cache on the big temp disk
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "WARN ") + "pip install " + " ".join(pkgs))

pip_install("hf_xet")   # faster Hugging Face downloads (harmless if already present)
pip_install("av")       # backup video decoder

# Same library version as the main study, so every model runs on identical library code.
# (Checked before transformers is imported, so no restart is needed.)
from importlib.metadata import version as _v, PackageNotFoundError
TRANSFORMERS_PIN = "5.16.1"
try:
    _have = _v("transformers")
except PackageNotFoundError:
    _have = None
if _have != TRANSFORMERS_PIN:
    print(f"transformers {_have} found -> installing {TRANSFORMERS_PIN} (the version used in the main study)")
    pip_install(f"transformers=={TRANSFORMERS_PIN}")
    assert _v("transformers") == TRANSFORMERS_PIN, "transformers pin failed — send me this log"
print("transformers", _v("transformers"), "| torch", _v("torch"), "| python", sys.version.split()[0])

if MODEL == "qwen3vl":
    from transformers import Qwen3VLForConditionalGeneration  # noqa: F401
else:
    from transformers import LlavaOnevisionForConditionalGeneration  # noqa: F401
print("transformers OK")

### 1b · Run record (+ resume from an earlier session)

In [ ]:
# ── Run record + resume from an earlier session ──────────────────────
import json, datetime, glob, shutil
RES_PATH = os.path.join(OUT_DIR, "results.jsonl")
ERR_PATH = os.path.join(OUT_DIR, "errors.log")
RUN_RECORD = {
    "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "benchmark": BENCHMARK, "model": MODEL_ID, "frames": NUM_FRAMES, "quick_test": QUICK_TEST, "study": "revision",
    "versions": {"transformers": _v("transformers"), "torch": _v("torch"), "python": sys.version.split()[0]},
    "analysis_plan": PREREG_URL, "sample_seed": SAMPLE_SEED,
    "videomme_chunks": VIDEOMME_CONFIRM_CHUNKS, "manifest_sha256": REP_MANIFEST[BENCHMARK]["sha256"],
    "methods": METHOD_GRID, "vcast": VCAST, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS,
    "noise": {"frame_shifts": NOISE_FRAME_SHIFTS, "option_orders": NOISE_OPTION_ORDERS},
}
with open(os.path.join(OUT_DIR, f"run_record_{datetime.datetime.now():%Y%m%d_%H%M%S}.json"), "w") as f:
    json.dump(RUN_RECORD, f, indent=2)

# If a previous session's results.jsonl for this benchmark is attached (Add Input), continue from it.
if not os.path.exists(RES_PATH):
    # matches e.g. .../flips_ext_qwen3vl_egoschema/results.jsonl (never the replication run's results)
    prev = [p for p in glob.glob("/kaggle/input/**/results.jsonl", recursive=True)
            if f"ext_{MODEL}_{BENCHMARK}" in p.lower().replace("-", "_") and "quick" not in p.lower()]
    if prev:
        shutil.copy(prev[0], RES_PATH)
        print("Resuming from", prev[0])
print(json.dumps(RUN_RECORD, indent=1)[:1500])

### 2 · Imports and GPU check

In [ ]:
# ── Imports + hardware check ──────────────────────────────────────────
import json, math, time, random, shutil, zipfile, gc, heapq
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from huggingface_hub import hf_hub_download, list_repo_files
import transformers
from transformers import AutoTokenizer

print("torch", torch.__version__, "| transformers", transformers.__version__)
N_GPU = torch.cuda.device_count()
assert N_GPU > 0 or DATA_CHECK_ONLY, "No GPU found! Right panel > Settings > Accelerator > 'GPU T4 x2', then Run All again."
for i in range(N_GPU):
    print(f"GPU{i}: {torch.cuda.get_device_name(i)}")
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
print(f"Free disk for temp videos: {free_gb:.1f} GB")
if free_gb < 12:
    print("⚠️ Less than 12 GB free — a 5 GB video chunk needs ~11 GB while unzipping.")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Optional: a Hugging Face token stored as a Kaggle Secret named HF_TOKEN (Add-ons > Secrets).
# Not needed for these public repos, but harmless.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets")
except Exception:
    print("No HF_TOKEN secret (fine for public repos)")

### 3 · Benchmarks: tables, the replication sample, downloads, frames

In [ ]:
# ── Data: benchmark tables, pre-registered sampling, video download, frame reading ──
import io, tarfile, hashlib, re, concurrent.futures as cf
from huggingface_hub import list_repo_files

LETTERS = "ABCDEFGH"
_PREFIX = re.compile(r"^\s*\(?[A-Ha-h][\.\):]\s*")
_norm = lambda s: re.sub(r"\s+", " ", str(s)).strip().rstrip(".").strip().lower()

def _opts(cands):
    """Plain candidate texts -> 'A. text' strings (any existing 'A.' / '(A)' prefix removed first)."""
    return [f"{LETTERS[i]}. {_PREFIX.sub('', str(c)).strip()}" for i, c in enumerate(cands)]

def _answer_letter(cands, answer):
    a = _norm(_PREFIX.sub("", str(answer)))
    hits = [i for i, c in enumerate(cands) if _norm(_PREFIX.sub("", str(c))) == a]
    return LETTERS[hits[0]] if len(hits) == 1 else None

def _hf(repo, fn, **kw):
    return hf_hub_download(repo, fn, repo_type="dataset", local_dir=os.path.join(WORK_DIR, repo.replace("/", "__")), **kw)

def _clean_cache():
    """Big archives leave a copy in the Hugging Face xet cache; remove it to keep disk free."""
    shutil.rmtree(os.path.join(os.environ.get("HF_HOME", ""), "xet"), ignore_errors=True)

def _sample(df, by, k, seed=SAMPLE_SEED):
    out = []
    for g, sub in df.groupby(by, sort=True):
        out.append(sub.sample(n=min(k, len(sub)), random_state=seed) if k else sub)
    return pd.concat(out).sort_values("qid", kind="stable").reset_index(drop=True)

# ═════ Video-MME (questions per chunk; videos come chunk by chunk) ═════
VMME_REPO = "lmms-eval/Video-MME"

def videomme_table():
    files = list_repo_files(VMME_REPO, repo_type="dataset")
    pq = [f for f in files if f.endswith(".parquet")]
    df = pd.concat([pd.read_parquet(_hf(VMME_REPO, f)) for f in pq], ignore_index=True)
    low = {c.lower(): c for c in df.columns}
    col = lambda *ns: next((low[n] for n in ns if n in low), None)
    out = pd.DataFrame({
        "video_key": df[col("videoid", "video_id")].astype(str),
        "video_alt": df[col("video_id", "videoid")].astype(str),
        "qid": df[col("question_id")].astype(str),
        "group": df[col("duration")].astype(str),
        "task": df[col("task_type")].astype(str),
        "question": df[col("question")].astype(str),
        "options": df[col("options")].apply(lambda o: [str(x) for x in list(o)]),
        "answer": df[col("answer")].astype(str).str.strip().str[0].str.upper(),
    })
    return out

VIDEO_EXT = (".mp4", ".mkv", ".webm", ".avi", ".mov")

def videomme_fetch_chunk(chunk):
    fn = f"videos_chunked_{chunk}.zip"
    t0 = time.time()
    zpath = _hf(VMME_REPO, fn)
    vdir = os.path.join(WORK_DIR, f"vmme_{chunk}")
    os.makedirs(vdir, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for m in z.infolist():
            if m.filename.lower().endswith(VIDEO_EXT):
                with z.open(m) as src, open(os.path.join(vdir, os.path.basename(m.filename)), "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 24)
    os.remove(zpath); _clean_cache()
    vids = {os.path.splitext(f)[0]: os.path.join(vdir, f) for f in os.listdir(vdir)}
    print(f"Video-MME chunk {chunk}: {len(vids)} videos in {time.time() - t0:.0f}s")
    return vids

# ═════ MVBench (video branch: pre-cut segments, one file per video) ═════
MVB_REPO, MVB_REV = "OpenGVLab/MVBench", "video"
MVB_FOLDERS = {   # lmms-eval DATA_LIST
    "object_interaction": "star/Charades_segment", "action_sequence": "star/Charades_segment",
    "action_prediction": "star/Charades_segment", "action_localization": "sta/sta_video_segment",
    "moving_count": "clevrer/video_validation", "character_order": "perception/videos",
    "object_shuffle": "perception/videos", "egocentric_navigation": "vlnqa",
    "moving_direction": "clevrer/video_validation", "episodic_reasoning": "tvqa/video_fps3_hq_segment",
    "fine_grained_action": "Moments_in_Time_Raw/videos", "scene_transition": "scene_qa/video",
    "state_change": "perception/videos", "moving_attribute": "clevrer/video_validation",
    "action_antonym": "ssv2_video_mp4", "unexpected_action": "FunQA_test/test",
    "counterfactual_inference": "clevrer/video_validation", "object_existence": "clevrer/video_validation",
    "action_count": "perception/videos",
}   # fine_grained_pose (NTU RGB+D) is excluded by the pre-registration

def _read_json_any(path):
    txt = open(path, encoding="utf-8").read().strip()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(l) for l in txt.splitlines() if l.strip()]

MVB_FILES = None
def mvbench_table():
    global MVB_FILES
    MVB_FILES = list_repo_files(MVB_REPO, repo_type="dataset", revision=MVB_REV)
    rows, dropped = [], 0
    for task, folder in MVB_FOLDERS.items():
        jf = next((f for f in MVB_FILES if f.endswith(f"json/{task}.json")), f"json/{task}.json")
        for i, d in enumerate(_read_json_any(_hf(MVB_REPO, jf, revision=MVB_REV))):
            ans = _answer_letter(d["candidates"], d["answer"])
            if ans is None or len(d["candidates"]) > len(LETTERS):
                dropped += 1
                continue
            rows.append({"qid": f"{task}-{i:04d}", "video_key": f"{folder}/{d['video']}", "video_alt": "",
                         "group": task, "task": task, "question": str(d["question"]),
                         "options": _opts(d["candidates"]), "answer": ans})
    print(f"MVBench: {len(rows)} questions in {len(MVB_FOLDERS)} tasks ({dropped} dropped: answer not among candidates)")
    return pd.DataFrame(rows)

def mvbench_fetch(keys):
    all_files, by_base = set(MVB_FILES), {}
    for f in MVB_FILES:
        by_base.setdefault(os.path.basename(f), []).append(f)
    def one(k):
        cand = [k] if k in all_files else [f for f in by_base.get(os.path.basename(k), [])
                                         if f.startswith(k.split("/")[0])] or by_base.get(os.path.basename(k), [])
        if not cand:
            raise FileNotFoundError(k)
        return k, _hf(MVB_REPO, cand[0], revision=MVB_REV)
    return _parallel_fetch(keys, one)

# ═════ EgoSchema (public 500-question subset) ═════
EGO_REPO = "lmms-lab/egoschema"
def egoschema_table():
    files = list_repo_files(EGO_REPO, repo_type="dataset")
    pq = [f for f in files if f.startswith("Subset/") and f.endswith(".parquet")]
    df = pd.concat([pd.read_parquet(_hf(EGO_REPO, f)) for f in pq], ignore_index=True)
    rows = []
    for _, d in df.iterrows():
        opts = [str(o) for o in list(d["option"])]
        rows.append({"qid": str(d["question_idx"]), "video_key": str(d["video_idx"]), "video_alt": "",
                     "group": "ego", "task": "ego", "question": str(d["question"]), "options": _opts(opts),
                     "answer": LETTERS[int(str(d["answer"]).strip())]})
    print(f"EgoSchema subset: {len(rows)} questions")
    return pd.DataFrame(rows)

def egoschema_fetch(keys):
    need, got = set(keys), {}
    files = sorted(f for f in list_repo_files(EGO_REPO, repo_type="dataset") if f.endswith(".zip"))
    vdir = os.path.join(WORK_DIR, "ego_videos"); os.makedirs(vdir, exist_ok=True)
    for fn in files:
        if not need - set(got):
            break
        t0 = time.time()
        zpath = _hf(EGO_REPO, fn)
        with zipfile.ZipFile(zpath) as z:
            for m in z.infolist():
                stem = os.path.splitext(os.path.basename(m.filename))[0]
                if stem in need and stem not in got and m.filename.lower().endswith(VIDEO_EXT):
                    dst = os.path.join(vdir, os.path.basename(m.filename))
                    with z.open(m) as src, open(dst, "wb") as out:
                        shutil.copyfileobj(src, out, 1 << 24)
                    got[stem] = dst
        os.remove(zpath); _clean_cache()
        print(f"  {fn}: {len(got)}/{len(need)} videos found ({time.time() - t0:.0f}s, "
              f"free disk {shutil.disk_usage(WORK_DIR).free / 1e9:.0f} GB)")
    return got

# ═════ LongVideoBench (validation, video only) ═════
LVB_REPO = "longvideobench/LongVideoBench"
def longvideobench_table():
    files = list_repo_files(LVB_REPO, repo_type="dataset")
    if "lvb_val.json" in files:
        data = _read_json_any(_hf(LVB_REPO, "lvb_val.json"))
    else:
        data = pd.read_parquet(_hf(LVB_REPO, next(f for f in files if f.startswith("validation")))).to_dict("records")
    rows = []
    for d in data:
        cands = d.get("candidates")
        if cands is None:
            cands = [d[f"option{i}"] for i in range(5) if d.get(f"option{i}") not in (None, "N/A")]
        cands = [str(c) for c in list(cands)]
        rows.append({"qid": str(d["id"]), "video_key": os.path.basename(str(d["video_path"])), "video_alt": "",
                     "group": str(d["duration_group"]), "task": str(d.get("question_category", "")),
                     "question": str(d["question"]), "options": _opts(cands),
                     "answer": LETTERS[int(d["correct_choice"])]})
    print(f"LongVideoBench val: {len(rows)} questions")
    return pd.DataFrame(rows)

class _PartsReader(io.RawIOBase):
    """Reads videos.tar.part.aa, .ab, ... as one stream; downloads each part when needed, deletes it after."""
    def __init__(self, parts):
        self.parts, self.k, self.f, self.path = list(parts), -1, None, None
    def readable(self):
        return True
    def _next(self):
        if self.f:
            self.f.close(); os.remove(self.path); _clean_cache()
        self.k += 1
        if self.k >= len(self.parts):
            self.f = None
            return False
        t0 = time.time()
        self.path = _hf(LVB_REPO, self.parts[self.k])
        self.f = open(self.path, "rb")
        print(f"  {self.parts[self.k]} ready ({time.time() - t0:.0f}s)")
        return True
    def readinto(self, b):
        while True:
            if self.f is None and not self._next():
                return 0
            n = self.f.readinto(b)
            if n:
                return n
            if not self._next():
                return 0

def longvideobench_fetch(keys):
    need, got = set(keys), {}
    parts = sorted(f for f in list_repo_files(LVB_REPO, repo_type="dataset") if f.startswith("videos.tar.part"))
    vdir = os.path.join(WORK_DIR, "lvb_videos"); os.makedirs(vdir, exist_ok=True)
    reader = io.BufferedReader(_PartsReader(parts), buffer_size=1 << 24)
    with tarfile.open(fileobj=reader, mode="r|") as tar:
        for m in tar:
            b = os.path.basename(m.name)
            if m.isfile() and b in need and b not in got:
                with tar.extractfile(m) as src, open(os.path.join(vdir, b), "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 24)
                got[b] = os.path.join(vdir, b)
            if not need - set(got):
                break
    return got

# ═════ MLVU (dev, the 7 multiple-choice tasks; one file per video) ═════
MLVU_REPO = "MLVU/MVLU"
MLVU_TASKS = ["1_plotQA", "2_needle", "3_ego", "4_count", "5_order", "6_anomaly_reco", "7_topic_reasoning"]
MLVU_FILES = None
def mlvu_table():
    global MLVU_FILES
    MLVU_FILES = list_repo_files(MLVU_REPO, repo_type="dataset")
    rows, dropped = [], 0
    for task in MLVU_TASKS:
        jf = next(f for f in MLVU_FILES if f.endswith(f"json/{task}.json"))
        for i, d in enumerate(_read_json_any(_hf(MLVU_REPO, jf))):
            ans = _answer_letter(d["candidates"], d["answer"])
            if ans is None or len(d["candidates"]) > len(LETTERS):
                dropped += 1
                continue
            rows.append({"qid": f"{task}-{i:04d}", "video_key": f"{task}/{d['video']}", "video_alt": "",
                         "group": task, "task": task, "question": str(d["question"]),
                         "options": _opts(d["candidates"]), "answer": ans})
    print(f"MLVU dev MC: {len(rows)} questions ({dropped} dropped)")
    return pd.DataFrame(rows)

def mlvu_fetch(keys):
    by_rel = {"/".join(f.split("/")[-2:]): f for f in MLVU_FILES if f.lower().endswith(VIDEO_EXT)}
    def one(k):
        if k not in by_rel:
            raise FileNotFoundError(k)
        return k, _hf(MLVU_REPO, by_rel[k])
    return _parallel_fetch(keys, one)

def _parallel_fetch(keys, one, workers=8):
    got, t0 = {}, time.time()
    with cf.ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(one, k): k for k in sorted(set(keys))}
        for i, fu in enumerate(cf.as_completed(futs)):
            try:
                k, p = fu.result()
                got[k] = p
            except Exception as e:
                with open(ERR_PATH, "a") as f:
                    f.write(f"{futs[fu]}\tdownload\t{type(e).__name__}: {e}\n")
            if (i + 1) % 100 == 0:
                print(f"  downloaded {i + 1}/{len(futs)} ({time.time() - t0:.0f}s)")
    print(f"Downloaded {len(got)}/{len(futs)} videos in {time.time() - t0:.0f}s")
    return got

# ═════ Manifest: the pre-registered sample, frozen to a file with its hash ═════
def _valid(df):
    """Pre-registration: questions with fewer than two options carry no information and are removed before sampling."""
    bad = df.options.apply(len) < 2
    if bad.any():
        print(f"Removed {int(bad.sum())} question(s) with fewer than two options: {df.qid[bad].tolist()}")
    return df[~bad]

def build_manifest():
    """The replication sample: the frozen question IDs in REP_MANIFEST (a subset of the main study's manifest)."""
    want = REP_MANIFEST[BENCHMARK]
    ids = sorted(want["qids"])
    assert hashlib.sha256("\n".join(ids).encode()).hexdigest() == want["sha256"], "replication manifest corrupted"
    full = {"videomme": videomme_table, "egoschema": egoschema_table,
            "longvideobench": longvideobench_table}[BENCHMARK]()
    df = _valid(full)
    df = df[df.qid.isin(set(ids))].sort_values("qid", kind="stable").reset_index(drop=True)
    missing = sorted(set(ids) - set(df.qid))
    if missing:
        raise RuntimeError(f"{len(missing)} replication questions not found in the benchmark table, e.g. {missing[:3]}")
    with open(os.path.join(OUT_DIR, f"manifest_{BENCHMARK}.json"), "w") as f:
        json.dump({"benchmark": BENCHMARK, "seed": SAMPLE_SEED, "n": len(ids), "sha256": want["sha256"],
                   "qids": ids}, f)
    print(f"Replication manifest: {len(df)} questions, {df.video_key.nunique()} videos, sha256 {want['sha256'][:16]}…")
    print("Per group:", df.group.value_counts().sort_index().to_dict())
    return df

# ═════ Frames ═════
def _frame_indices(total, n, shift):
    """Uniform sampling as in LMMs-Eval (np.linspace over all frames). shift = fraction of one sampling
    interval to move every sample later (noise passes)."""
    idx = np.linspace(0, total - 1, n)
    if shift:
        idx = idx + shift * (total - 1) / max(n - 1, 1)
    return np.clip(idx, 0, total - 1).round().astype(int)

def _read_cv2(path, n, shift=0.0, meta=None):
    import cv2
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0) or None
    frames, used = [], []
    if total > 0:
        for k in _frame_indices(total, n, shift):
            ok = False
            for off in (0, -1, 1, -5, 5, -20):
                kk = int(np.clip(k + off, 0, total - 1))
                cap.set(cv2.CAP_PROP_POS_FRAMES, kk)
                ok, fr = cap.read()
                if ok:
                    break
            if ok:
                frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); used.append(kk)
    cap.release()
    if meta is not None:
        meta.update(fps=fps, total=total, indices=used)
    return frames

def _read_av(path, n, shift=0.0, meta=None):
    import av
    out, used = [], []
    with av.open(path) as c:
        s = c.streams.video[0]
        dur = float(s.duration * s.time_base) if s.duration else float(c.duration or 0) / 1e6
        fps = float(s.average_rate) if s.average_rate else None
        end = max(dur - 0.5, 0)
        ts = np.linspace(0, end, n)
        if shift:
            ts = np.clip(ts + shift * end / max(n - 1, 1), 0, end)
        for t in ts:
            c.seek(int(t / s.time_base), stream=s, any_frame=False, backward=True)
            for fr in c.decode(s):
                out.append(fr.to_ndarray(format="rgb24")); used.append(int(round(t * (fps or 24))))
                break
    if meta is not None:
        meta.update(fps=fps, total=int(round(dur * (fps or 24))), indices=used)
    return out

def read_frames(path, n, shift=0.0, meta=None):
    """n frames sampled uniformly; meta (optional dict) receives fps, total frame count and the frame indices used."""
    frames = []
    try:
        frames = _read_cv2(path, n, shift, meta)
    except Exception as e:
        print("cv2 failed:", e)
    if len(frames) < n:
        try:
            frames = _read_av(path, n, shift, meta)
        except Exception as e:
            print("av failed:", e)
    if len(frames) == 0:
        raise RuntimeError(f"Could not decode {path}")
    if meta is not None:
        idx = list(meta.get("indices", []))[:len(frames)]
        while len(idx) < n:
            idx.append(idx[-1] if idx else 0)
        meta["indices"] = idx[:n]
    while len(frames) < n:
        frames.append(frames[-1])
    return frames[:n]

def read_all_frame_sets(path):
    """The three frame sets every video needs: original, and the two shifted noise passes (with their metadata)."""
    out, metas = {}, {}
    for k, s in [("orig", 0.0)] + list(NOISE_FRAME_SHIFTS.items()):
        metas[k] = {}
        out[k] = read_frames(path, NUM_FRAMES, shift=s, meta=metas[k])
    out["_meta"] = metas
    return out

# ═════ Option-order noise passes ═════
def _new_index(kind, j, n):
    """Which ORIGINAL option is shown at new position j."""
    return (j + 1) % n if kind == "option_cycle" else n - 1 - j

def reorder_options(options, kind):
    body = [_PREFIX.sub("", str(o)).strip() for o in options]
    n = len(body)
    return [f"{LETTERS[j]}. {body[_new_index(kind, j, n)]}" for j in range(n)]

def map_back(res, n, kind):
    """Express a reordered result in the original option letters."""
    lp_new = np.array(res["logp"])
    lp = np.empty(n)
    for j in range(n):
        lp[_new_index(kind, j, n)] = lp_new[j]
    srt = np.sort(lp)[::-1]
    out = dict(res)
    out.update(pred=LETTERS[int(lp.argmax())], logp=lp.round(4).tolist(), margin=float(srt[0] - srt[1]))
    return out

### 4 · Model, T4 memory fix, our decoder loop, prefix caching

In [ ]:
# ── Model: Qwen3-VL-8B-Instruct with direct control over visual tokens ─────
# ── T4 memory fix + safe attention routing ─────────────────────────────
# 1) Expand grouped K/V heads ourselves so PyTorch can use the memory-efficient kernel (math kernel -> OOM on T4).
# 2) Never hand an explicit mask to the memory-efficient kernel: on T4 that path can crash with
#    "illegal memory access". A plain causal mask is converted to is_causal=True (identical maths).
# 3) Guard: if any NON-causal mask (e.g. transformers mistaking position gaps for packed sequences)
#    reaches a long attention call, stop loudly instead of computing something silently wrong.
from torch.nn.attention import sdpa_kernel, SDPBackend
if not hasattr(torch.nn.functional, "_astra_orig_sdpa"):
    torch.nn.functional._astra_orig_sdpa = torch.nn.functional.scaled_dot_product_attention
_orig_sdpa = torch.nn.functional._astra_orig_sdpa
_TRIL_CACHE = {}
def _plain_causal(mask, L, S):
    if L != S or mask.dim() != 4 or mask.shape[0] != 1 or mask.shape[1] != 1:
        return False
    key = (L, str(mask.device))
    if key not in _TRIL_CACHE:
        _TRIL_CACHE.clear()
        _TRIL_CACHE[key] = torch.ones(L, L, dtype=torch.bool, device=mask.device).tril()
    allowed = mask[0, 0] if mask.dtype == torch.bool else (mask[0, 0] == 0)
    return torch.equal(allowed, _TRIL_CACHE[key])

def _sdpa_safe(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, enable_gqa=False, **kw):
    if enable_gqa and key.shape[-3] != query.shape[-3]:
        rep = query.shape[-3] // key.shape[-3]
        key = key.repeat_interleave(rep, dim=-3)
        value = value.repeat_interleave(rep, dim=-3)
    if attn_mask is not None:
        L, S = query.shape[-2], key.shape[-2]
        if _plain_causal(attn_mask, L, S):
            attn_mask, is_causal = None, True
        elif L > 1024:
            raise RuntimeError("A non-causal attention mask reached a long attention call — stopping instead of "
                               "computing silently wrong results. Send me this error.")
        elif query.dtype == torch.float16:
            # short question block over a cached prefix: compute in fp32 (cheap at this size), so cached passes
            # match the fp32-accumulating kernel used for full passes as closely as possible
            m = attn_mask if attn_mask.dtype == torch.bool else attn_mask.float()
            return _orig_sdpa(query.float(), key.float(), value.float(), attn_mask=m, dropout_p=dropout_p,
                              is_causal=is_causal, scale=scale, **kw).to(query.dtype)
    return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale, **kw)

torch.nn.functional.scaled_dot_product_attention = _sdpa_safe
NO_MATH = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION]
print("T4 attention fix applied (K/V heads expanded, causal masks routed to is_causal, math kernel disabled for the LLM)")


# Qwen3-VL specifics handled here: (1) 2 frames form one temporal patch, so 32 frames give 16 temporal groups, each
# introduced by a timestamp "<t seconds>" and wrapped in vision-start/end tokens; (2) 3-D (time, height, width) rotary
# positions; (3) DeepStack: features from three vision layers are added to the visual positions after LLM layers 0-2.
# Compression keeps each kept token's own 3-D position and its own DeepStack features (as FlashVID's official code).
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from transformers.models.qwen3_vl.modeling_qwen3_vl import apply_rotary_pos_emb, apply_rotary_pos_emb_vision, repeat_kv
from transformers.video_utils import VideoMetadata
import torchvision.transforms.functional as TVF

max_memory = {i: "13GiB" for i in range(N_GPU)}
if N_GPU >= 2:
    max_memory[0] = "9GiB"
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", max_memory=max_memory, attn_implementation="sdpa", dtype=torch.float16).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer

def _get(root, paths):
    for p in paths:
        obj, ok = root, True
        for part in p.split("."):
            if hasattr(obj, part):
                obj = getattr(obj, part)
            else:
                ok = False
                break
        if ok:
            return obj
    raise AttributeError(f"None of {paths} found — send me the error and transformers version")

visual  = _get(model, ["model.visual", "visual"])
body    = _get(model, ["model.language_model", "language_model"])
lm_head = _get(model, ["lm_head"])
CFG, TCFG, VCFG = model.config, model.config.text_config, model.config.vision_config
VIDEO_TOKEN_ID = CFG.video_token_id
VSTART_ID, VEND_ID = CFG.vision_start_token_id, CFG.vision_end_token_id
MERGE = VCFG.spatial_merge_size
FACTOR = VCFG.patch_size * MERGE                         # 32 pixels per merged token side
NLAYERS, HID = TCFG.num_hidden_layers, TCFG.hidden_size
NHEADS, NKV = TCFG.num_attention_heads, TCFG.num_key_value_heads
HEAD_DIM = getattr(TCFG, "head_dim", None) or HID // NHEADS
INTER = TCFG.intermediate_size
EMB_DEVICE = body.embed_tokens.weight.device
LM_DEVICE = lm_head.weight.device
VDEV = next(visual.parameters()).device
N_DEEPSTACK = len(VCFG.deepstack_visual_indexes)
print(f"LLM layers={NLAYERS} hidden={HID} heads={NHEADS}/{NKV} head_dim={HEAD_DIM} | DeepStack layers {N_DEEPSTACK} | "
      f"{FACTOR}px per visual token, 2 frames per temporal group")

LETTERS = "ABCDEFGH"
LETTER_IDS = []
for L in LETTERS:
    ids = tok.encode(L, add_special_tokens=False)
    assert len(ids) == 1, f"Letter {L} is not a single token"
    LETTER_IDS.append(ids[0])

PREFIX = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n"

def _letter_list(n):
    L = list(LETTERS[:n])
    return L[0] if n == 1 else (f"{L[0]} or {L[1]}" if n == 2 else ", ".join(L[:-1]) + ", or " + L[-1])

def suffix_text(question, options):
    opts = "\n".join(str(o) for o in options)
    return ("\nSelect the best answer to the following multiple-choice question based on the video. "
            f"Respond with only the letter ({_letter_list(len(options))}) of the correct option.\n"
            f"{question}\n{opts}\nThe best answer is:<|im_end|>\n<|im_start|>assistant\n")

def build_suffix_ids(question, options):
    """The main study's prompt (identical text)."""
    return tok(suffix_text(question, options), add_special_tokens=False, return_tensors="pt").input_ids[0]

def smart_resize_frame(h, w, factor=None, min_pixels=None, max_pixels=None):
    """qwen-vl-utils smart_resize (per frame), with FlashVID's baseline pixel limits."""
    factor = factor or FACTOR
    min_pixels, max_pixels = min_pixels or MIN_PIXELS, max_pixels or MAX_PIXELS
    h_bar = max(factor, round(h / factor) * factor)
    w_bar = max(factor, round(w / factor) * factor)
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((h * w) / max_pixels)
        h_bar = max(factor, math.floor(h / beta / factor) * factor)
        w_bar = max(factor, math.floor(w / beta / factor) * factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (h * w))
        h_bar = math.ceil(h * beta / factor) * factor
        w_bar = math.ceil(w * beta / factor) * factor
    return h_bar, w_bar

def prepare_video(frames, meta, prompt_tail=""):
    """Resize (bicubic, antialiased, as qwen-vl-utils) and run the official processor (patchify, normalise,
    timestamps). Returns the processor outputs for the text PREFIX + video (+ prompt_tail)."""
    h, w = frames[0].shape[:2]
    H, W = smart_resize_frame(h, w)
    v = torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2)
    v = TVF.resize(v, [H, W], interpolation=TVF.InterpolationMode.BICUBIC, antialias=True)
    v = v.round().clamp(0, 255).to(torch.uint8).permute(0, 2, 3, 1).numpy()
    fps = (meta or {}).get("fps") or 24.0
    idx = list((meta or {}).get("indices") or range(len(frames)))
    md = VideoMetadata(total_num_frames=int((meta or {}).get("total") or (max(idx) + 1)), fps=float(fps),
                       frames_indices=idx, width=W, height=H)
    return processor(text=[PREFIX + processor.video_token + prompt_tail], videos=[v], video_metadata=[md],
                     do_resize=False, do_sample_frames=False, cap_pixels_per_frame=False, return_tensors="pt")

# ── FlashVID needs the last vision block's attention (mean over heads and queries, per temporal group) ──
_LAST_ATTN = {}
def _capture_last_attn(mod, args, kwargs):
    hs = args[0] if args else kwargs["hidden_states"]
    _LAST_ATTN["h"] = hs
    _LAST_ATTN["pe"] = kwargs.get("position_embeddings")
    _LAST_ATTN["cu"] = kwargs.get("cu_seqlens")
_hook = visual.blocks[-1].attn.register_forward_pre_hook(_capture_last_attn, with_kwargs=True)

@torch.no_grad()
def _flashvid_cls_attention(attn_mod, n_groups):
    """FlashVID Qwen3VLVisionAttention_forward(return_logits=True) + the pooling in its vision forward."""
    hs, (cos, sin) = _LAST_ATTN["h"], _LAST_ATTN["pe"]
    seq = hs.shape[0]
    q, k, _ = attn_mod.qkv(hs).reshape(seq, 3, attn_mod.num_heads, -1).permute(1, 0, 2, 3).unbind(0)
    q, k = apply_rotary_pos_emb_vision(q, k, cos, sin)
    q = q.transpose(0, 1).reshape(attn_mod.num_heads, n_groups, -1, attn_mod.head_dim).permute(1, 0, 2, 3)
    k = k.transpose(0, 1).reshape(attn_mod.num_heads, n_groups, -1, attn_mod.head_dim).permute(1, 0, 2, 3)
    out = []
    for i in range(n_groups):
        a = torch.matmul(q[i], k[i].transpose(-1, -2)) / attn_mod.head_dim ** 0.5
        a = torch.softmax(a, dim=-1, dtype=torch.float32).to(q.dtype)
        out.append(a.mean(0).mean(0))
    w = torch.stack(out, 0)                                   # (groups, patches per group)
    return w.view(n_groups, w.shape[-1] // (MERGE * MERGE), -1).mean(-1)

@torch.no_grad()
def encode_frames(frames, meta=None):
    """One frame set -> everything the passes need: the prefix token ids (with timestamps), their 3-D positions,
    the merged visual tokens (N, D), DeepStack features (3 x (N, D)), FlashVID's per-group attention (G, P)."""
    inp = prepare_video(frames, meta)
    ids = inp["input_ids"][0]
    grid = inp["video_grid_thw"]
    G, Hp, Wp = [int(x) for x in grid[0]]
    out = visual(inp["pixel_values_videos"].to(VDEV, torch.float16), grid_thw=grid.to(VDEV), return_dict=True)
    X = out.pooler_output if hasattr(out, "pooler_output") else out[0]
    DS = list(out.deepstack_features)
    CA = _flashvid_cls_attention(visual.blocks[-1].attn, G)
    _LAST_ATTN.clear()
    pos, _ = model.model.get_rope_index(ids[None], inp["mm_token_type_ids"], video_grid_thw=grid)
    pos = pos[:, 0]                                           # (3, L)
    vmask = ids == VIDEO_TOKEN_ID
    P = (Hp // MERGE) * (Wp // MERGE)
    assert int(vmask.sum()) == X.shape[0] == G * P, "visual token count mismatch"
    text_emb = body.embed_tokens(ids.to(EMB_DEVICE))
    return {"ids": ids, "pos": pos, "vmask": vmask, "X": X.to(EMB_DEVICE), "DS": [d.to(EMB_DEVICE) for d in DS],
            "CA": CA.to(EMB_DEVICE).float(), "G": G, "Hm": Hp // MERGE, "Wm": Wp // MERGE, "P": P,
            "text_emb": text_emb, "next_pos": int(pos.max()) + 1, "n_full": int(X.shape[0]),
            "pixel_values_videos": inp["pixel_values_videos"], "grid": grid}

def assemble(enc, groups, drop_start_end=False):
    """Builds a prefix from per-group visual tokens. groups[g] = (feat (k, D), [ds (k, D)] * 3, pos (3, k)).
    Text tokens (prompt, timestamps, vision start/end) keep their own embeddings and positions."""
    ids, pos, vm = enc["ids"], enc["pos"], enc["vmask"]
    dev = enc["text_emb"].device
    keep_text = ~vm
    if drop_start_end:
        keep_text &= (ids != VSTART_ID) & (ids != VEND_ID)
    vid_idx = torch.nonzero(vm).flatten()
    starts = vid_idx[::enc["P"]].tolist()                    # first index of each temporal group
    emb, pp, vmask, ds = [], [], [], [[] for _ in range(N_DEEPSTACK)]
    cur = 0
    for g in range(enc["G"]):
        s = starts[g]
        t = torch.arange(cur, s)
        t = t[keep_text[cur:s]]
        emb.append(enc["text_emb"][t.to(dev)]); pp.append(pos[:, t]); vmask.append(torch.zeros(len(t), dtype=torch.bool))
        f, dsl, gp = groups[g]
        emb.append(f.to(dev, enc["text_emb"].dtype)); pp.append(gp.cpu()); vmask.append(torch.ones(f.shape[0], dtype=torch.bool))
        for j in range(N_DEEPSTACK):
            ds[j].append(dsl[j])
        cur = s + enc["P"]
    t = torch.arange(cur, len(ids))
    t = t[keep_text[cur:]]
    emb.append(enc["text_emb"][t.to(dev)]); pp.append(pos[:, t]); vmask.append(torch.zeros(len(t), dtype=torch.bool))
    return {"embeds": torch.cat(emb, 0), "pos": torch.cat(pp, 1), "vmask": torch.cat(vmask, 0),
            "ds": [torch.cat(d, 0) for d in ds], "next_pos": enc["next_pos"]}

def _group_pos(enc):
    return enc["pos"][:, enc["vmask"]].view(3, enc["G"], enc["P"])

def full_spec(enc):
    G, P = enc["G"], enc["P"]
    gp = _group_pos(enc)
    X, DS = enc["X"].view(G, P, -1), [d.view(G, P, -1) for d in enc["DS"]]
    s = assemble(enc, [(X[g], [d[g] for d in DS], gp[:, g]) for g in range(G)])
    s.update(inner=None, nvis=enc["n_full"])
    return s

def llm_flops_layers(ns):
    """Prefill FLOPs of the LLM when layer l sees ns[l] tokens (q/o projections use heads x head_dim)."""
    dkv, dq = NKV * HEAD_DIM, NHEADS * HEAD_DIM
    return float(sum(2 * (2 * n * HID * dq + 2 * n * HID * dkv + 2 * n * n * dq + 3 * n * HID * INTER) for n in ns))

def llm_flops(n):
    return llm_flops_layers([n] * NLAYERS)

def _sync():
    for i in range(N_GPU):
        torch.cuda.synchronize(i)

@torch.no_grad()
def last_query_attention(layer, h, pe):
    """Attention of the last token over all tokens in this layer (FlashVID's output_attentions fallback)."""
    a = layer.self_attn
    dev = a.q_proj.weight.device
    h = h.to(dev)
    cos, sin = pe[0].to(dev), pe[1].to(dev)
    hn = layer.input_layernorm(h)
    q = a.q_norm(a.q_proj(hn[:, -1:]).view(1, 1, -1, HEAD_DIM)).transpose(1, 2)
    k = a.k_norm(a.k_proj(hn).view(1, hn.shape[1], -1, HEAD_DIM)).transpose(1, 2)
    q, _ = apply_rotary_pos_emb(q, q, cos[:, -1:], sin[:, -1:])
    _, k = apply_rotary_pos_emb(k, k, cos, sin)
    k = repeat_kv(k, NHEADS // NKV)
    w = torch.matmul(q, k.transpose(2, 3)) / HEAD_DIM ** 0.5
    return torch.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)

@torch.no_grad()
def apply_inner(h, pe, attn, inner, vmask):
    """FlashVID utils.fastv_prune with visual_pos_masks (its Qwen3-VL path)."""
    dev = h.device
    vm = vmask.to(dev)
    n_vis = int(vm.sum())
    n_keep = math.ceil(n_vis * inner["keep_ratio"])
    a = attn.mean(dim=1)[0, -1].to(dev)[vm]
    vis_idx = torch.nonzero(vm).flatten()
    top = torch.topk(a, k=n_keep).indices.sort().values
    keep = torch.sort(torch.cat([torch.nonzero(~vm).flatten(), vis_idx[top]])).values
    pe = (pe[0][..., keep.to(pe[0].device), :], pe[1][..., keep.to(pe[1].device), :])
    return h[:, keep], pe, n_keep

@torch.no_grad()
def forward_layers(x, pos3, vmask, ds, inner=None, nvis0=0):
    """Our own decoder loop (same calls as Qwen3VLTextModel.forward, incl. DeepStack); lets FlashVID prune inside."""
    pe = body.rotary_emb(x, pos3)
    h, attn, ns, nvis = x, None, [], []
    vis_now = nvis0
    vm_dev = vmask.to(x.device)
    for li, layer in enumerate(body.layers):
        if inner is not None and li == inner["layer"]:
            h, pe, vis_now = apply_inner(h, pe, attn, inner, vm_dev)
            attn = None
        if inner is not None and li == inner["layer"] - 1:
            attn = last_query_attention(layer, h, pe)
        ns.append(h.shape[1]); nvis.append(vis_now)
        h = layer(h, position_embeddings=pe, attention_mask=None, position_ids=None, use_cache=False)
        if isinstance(h, tuple):
            h = h[0]
        if ds is not None and li < len(ds):
            hd = h.device
            h = h.clone()
            h[0, vm_dev.to(hd)] = h[0, vm_dev.to(hd)] + ds[li].to(hd, h.dtype)
    h = body.norm(h.to(body.norm.weight.device))
    return h, ns, nvis

def _result(logits, n_opt, extra):
    lp = torch.log_softmax(logits[LETTER_IDS[:n_opt]], -1).cpu().numpy()
    if not np.all(np.isfinite(lp)):
        raise FloatingPointError("NaN/inf logits (fp16 overflow)")
    srt = np.sort(lp)[::-1]
    return {"pred": LETTERS[int(lp.argmax())], "logp": lp.round(4).tolist(), "margin": float(srt[0] - srt[1]), **extra}

def _seq(spec, suf_ids=None):
    E = body.embed_tokens
    x, pos, vm = spec["embeds"], spec["pos"], spec["vmask"]
    if suf_ids is not None:
        suf = E(suf_ids.to(EMB_DEVICE))
        n_q = suf.shape[0]
        x = torch.cat([x, suf.to(x.dtype)], 0)
        pos = torch.cat([pos, (spec["next_pos"] + torch.arange(n_q)).view(1, -1).expand(3, -1)], 1)
        vm = torch.cat([vm, torch.zeros(n_q, dtype=torch.bool)], 0)
    return x[None].to(EMB_DEVICE), pos[:, None, :].to(EMB_DEVICE), vm

@torch.no_grad()
def llm_pass(spec, suf_ids, n_opt, manual=False):
    x, pos3, vm = _seq(spec, suf_ids)
    for i in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(i)
    _sync(); t0 = time.time()
    inner = spec.get("inner")
    with sdpa_kernel(NO_MATH):
        if inner is None and not manual:
            out = body(inputs_embeds=x, position_ids=pos3, visual_pos_masks=vm[None].to(EMB_DEVICE),
                       deepstack_visual_embeds=spec["ds"], use_cache=False)
            h = out[0][0, -1]
            ns, nvis = [x.shape[1]] * NLAYERS, [spec["nvis"]] * NLAYERS
        else:
            hh, ns, nvis = forward_layers(x, pos3, vm, spec["ds"], inner, nvis0=spec["nvis"])
            h = hh[0, -1]
    logits = lm_head(h.to(LM_DEVICE)[None])[0].float()
    _sync(); dt = time.time() - t0
    return _result(logits, n_opt, {"ntok": int(spec["nvis"]), "vis_mean_layers": float(np.mean(nvis)), "sec": float(dt),
                                   "flops": llm_flops_layers(ns),
                                   "peak_gb": round(max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1e9, 3)})

# ── Prefix caching ──
from transformers import DynamicCache

def _new_cache():
    try:
        return DynamicCache(config=TCFG)
    except TypeError:
        return DynamicCache()

def _crop(cache, n):
    extra = cache.get_seq_length() - n
    if extra > 0 and hasattr(cache, "crop"):
        cache.crop(-extra)
    if cache.get_seq_length() != n:
        for l in cache.layers:
            l.keys, l.values = l.keys[..., :n, :], l.values[..., :n, :]
    assert cache.get_seq_length() == n, "could not restore the cached prefix"

@torch.no_grad()
def prefill(spec):
    x, pos3, vm = _seq(spec)
    for i in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(i)
    cache = _new_cache()
    _sync(); t0 = time.time()
    with sdpa_kernel(NO_MATH):
        body(inputs_embeds=x, position_ids=pos3, visual_pos_masks=vm[None].to(EMB_DEVICE),
             deepstack_visual_embeds=spec["ds"], past_key_values=cache, use_cache=True)
    _sync()
    return {"cache": cache, "n": int(x.shape[1]), "next_pos": spec["next_pos"], "sec": time.time() - t0,
            "nvis": int(spec["nvis"]), "peak": max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1e9}

@torch.no_grad()
def suffix_pass(P, suf_ids, n_opt):
    E = body.embed_tokens
    suf = E(suf_ids.to(EMB_DEVICE))[None]
    n_q = suf.shape[1]
    pos3 = (P["next_pos"] + torch.arange(n_q, device=EMB_DEVICE)).view(1, 1, -1).expand(3, 1, -1)
    am = torch.ones(1, P["n"] + n_q, dtype=torch.long, device=EMB_DEVICE)
    _sync(); t0 = time.time()
    with sdpa_kernel([SDPBackend.MATH]):
        out = body(inputs_embeds=suf, attention_mask=am, position_ids=pos3, past_key_values=P["cache"], use_cache=True)
    h = out[0][0, -1]
    logits = lm_head(h.to(LM_DEVICE)[None])[0].float()
    _sync(); dt = time.time() - t0
    _crop(P["cache"], P["n"])
    return _result(logits, n_opt, {"ntok": P["nvis"], "vis_mean_layers": float(P["nvis"]), "sec": float(P["sec"] + dt),
                                   "sec_suffix": float(dt), "flops": llm_flops(P["n"] + n_q),
                                   "peak_gb": round(P["peak"], 3)})

@torch.no_grad()
def preflight_attention(n_tok):
    dev = body.layers[0].self_attn.q_proj.weight.device
    q = torch.randn(1, NHEADS, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    kv = torch.randn(1, NKV, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    mask = torch.ones(n_tok, n_tok, dtype=torch.bool, device=dev).tril()[None, None]
    torch.cuda.reset_peak_memory_stats(dev)
    base = torch.cuda.memory_allocated(dev)
    try:
        with sdpa_kernel(NO_MATH):
            o1 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, is_causal=True, enable_gqa=True)
            o2 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, attn_mask=mask, enable_gqa=True)  # routed to is_causal
        torch.cuda.synchronize(dev)          # surface any asynchronous CUDA error HERE, not in a later cell
        assert torch.equal(o1, o2), "causal-mask routing changed the result"
    except RuntimeError as e:
        raise RuntimeError(f"Memory-efficient attention is not available here ({e}). Send me this message.")
    del o1, o2
    peak = (torch.cuda.max_memory_allocated(dev) - base) / 1e9
    del q, kv, mask
    torch.cuda.empty_cache()
    for i in range(N_GPU):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU{i}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
    print(f"Attention preflight OK ✅  {n_tok} tokens -> extra peak memory {peak:.2f} GB")

preflight_attention(NUM_FRAMES // 2 * (MAX_PIXELS // FACTOR ** 2) + 600)
print("Model ready ✅")

### 4b · Safety checks (run on the first video)

In [ ]:
# ── Safety check 1: our pipeline must reproduce the official model forward (processor + Qwen3-VL forward) ──
def equivalence_check(frame_sets, row):
    report = {"transformers": transformers.__version__}
    try:
        frames, meta = frame_sets["orig"], frame_sets["_meta"]["orig"]
        n_opt = min(len(row.options), len(LETTERS))
        inp = prepare_video(frames, meta, prompt_tail=suffix_text(row.question, row.options))
        dev_in = {k: (v.to(VDEV, torch.float16) if torch.is_floating_point(v) else v.to(EMB_DEVICE)) for k, v in inp.items()}
        with torch.no_grad(), sdpa_kernel(NO_MATH):
            try:
                out = model(**dev_in, logits_to_keep=1)
            except TypeError:
                out = model(**dev_in, num_logits_to_keep=1)
        lp_off = torch.log_softmax(out.logits[0, -1].float()[LETTER_IDS[:n_opt]], -1).cpu().numpy()
        del out, dev_in
        enc = encode_frames(frames, meta)
        spec = full_spec(enc)
        suf = build_suffix_ids(row.question, row.options)
        report["prompt_tokens_identical"] = bool(torch.equal(inp["input_ids"][0], torch.cat([enc["ids"], suf])))
        report["n_video_tokens"] = int(enc["n_full"])
        report["temporal_groups"] = int(enc["G"]); report["tokens_per_group"] = int(enc["P"])
        ours = llm_pass(spec, suf, n_opt)
        man = llm_pass(spec, suf, n_opt, manual=True)
        report["logp_official"] = lp_off.round(4).tolist()
        report["logp_ours"] = ours["logp"]
        report["max_abs_gap"] = float(np.abs(lp_off - np.array(ours["logp"])).max())
        report["logp_ours_manual_loop"] = man["logp"]
        report["max_abs_gap_manual_loop"] = float(np.abs(np.array(man["logp"]) - np.array(ours["logp"])).max())
        report["full_model_answer"] = ours["pred"]
        ok = report["prompt_tokens_identical"] and report["max_abs_gap"] <= 0.15 and report["max_abs_gap_manual_loop"] <= 0.05
        report["status"] = "PASS" if ok else "FAIL"
    except Exception as e:
        report["status"] = "SKIPPED"
        report["error"] = f"{type(e).__name__}: {e}"
    with open(os.path.join(OUT_DIR, "equivalence_check.json"), "w") as f:
        json.dump(report, f, indent=2)
    print("Equivalence check:", json.dumps(report, indent=2))
    if report["status"] != "PASS":
        raise RuntimeError("Equivalence check failed — stopping so no GPU hours are wasted. "
                           "Send me equivalence_check.json and the log.")
    gc.collect(); torch.cuda.empty_cache()


# ── Safety check 2: prefix caching must give the same answers as running each pass in full ──
def cache_check(enc, comp, rows):
    report = {"checks": []}
    picks = [("full", None)] + [k for k in comp if k in (("vcast", "0.1"),)]
    for key in picks:
        spec = full_spec(enc) if key[0] == "full" else comp[key]
        P = prefill(spec)
        for row in rows[:2]:
            n_opt = min(len(row.options), len(LETTERS))
            for opts in (row.options, reorder_options(row.options, "option_reverse")):
                suf = build_suffix_ids(row.question, opts)
                a = llm_pass(spec, suf, n_opt)
                b = suffix_pass(P, suf, n_opt)
                report["checks"].append({"pass": "/".join(map(str, key)) if key[0] != "full" else "full",
                                         "qid": row.qid, "same_answer": a["pred"] == b["pred"],
                                         "gap": float(np.abs(np.array(a["logp"]) - np.array(b["logp"])).max()),
                                         "speedup_per_extra_question": round(a["sec"] / max(b["sec_suffix"], 1e-6), 1)})
        del P
    same = np.mean([c["same_answer"] for c in report["checks"]])
    report["max_gap"] = max(c["gap"] for c in report["checks"])
    report["same_answer_share"] = float(same)
    report["status"] = "PASS" if (same == 1.0 and report["max_gap"] <= 0.15) else "FAIL"
    with open(os.path.join(OUT_DIR, "cache_check.json"), "w") as f:
        json.dump(report, f, indent=2)
    print("Cache check:", report["status"], f"| max log-prob gap {report['max_gap']:.4f} | "
          f"same answers {100 * same:.0f}% | an extra question on a cached prefix is "
          f"~{np.median([c['speedup_per_extra_question'] for c in report['checks']]):.0f}x faster than a full pass")
    if report["status"] != "PASS":
        raise RuntimeError("Cache check failed — stopping so no GPU hours are wasted. Send me cache_check.json.")
    gc.collect(); torch.cuda.empty_cache()

### 5 · V-CAST with the official Qwen3-VL code (+ self-test)

In [ ]:
# ══ V-CAST (Lin et al., NeurIPS 2026) — github.com/xinyouu/V-CAST, compressor/v_cast/modeling_qwen3_vl_v_cast.py ══
# MIT License (repository LICENSE). The functions between the two rulers are copied verbatim from the official file
# (commit f15d276, 3 Oct 2026). As in the official Qwen3-VL path, the merged tokens of each temporal group are one
# "frame", the selected tokens keep their own 3-D positions and DeepStack features, and only video tokens are
# removed (vision start/end and timestamp tokens stay). The official code fixes the ratio at 25%; we pass 5%, 10%, 25%.
from typing import List, Optional

# ─────────────────────────────── copied verbatim from V-CAST ───────────────────────────────
DEFAULT_RETAIN_RATIO = 0.25
DEFAULT_MIN_K = 1
DEFAULT_BUDGET_TEMP = 0.7


def _v_cast_print() -> bool:
    return True


def _tokens_per_frame(grid_thw_row: torch.Tensor, spatial_merge_size: int) -> tuple[int, int, int, int]:
    t = int(grid_thw_row[0].item())
    h = int(grid_thw_row[1].item())
    w = int(grid_thw_row[2].item())
    s = max(1, int(spatial_merge_size))
    resize_h = max(1, h // s)
    resize_w = max(1, w // s)
    return t, int(resize_h), int(resize_w), int(resize_h * resize_w)


def _compute_curvature(frame_reps: torch.Tensor) -> torch.Tensor:
    """
    frame_reps: [T, D], normalized
    returns curvature: [T]
    """
    t = int(frame_reps.shape[0])
    if t <= 1:
        return torch.ones((t,), device=frame_reps.device, dtype=torch.float32)
    v_in = frame_reps[1:-1] - frame_reps[:-2]
    v_out = frame_reps[2:] - frame_reps[1:-1]
    curv = 1.0 - F.cosine_similarity(v_in, v_out, dim=-1, eps=1e-6)
    ones = torch.ones((1,), device=frame_reps.device, dtype=curv.dtype)
    return torch.cat([ones, curv, ones], dim=0)


def _allocate_budget_per_frame(
    curvature: torch.Tensor,
    total_budget: int,
    *,
    min_k: int,
    max_k: int,
) -> torch.Tensor:
    """
    Allocate per-frame budgets summing to total_budget.
    curvature: [T] float
    returns: [T] long
    """
    t = int(curvature.shape[0])
    if t <= 0:
        return torch.zeros((0,), device=curvature.device, dtype=torch.long)

    total_budget = int(max(0, min(int(total_budget), int(t * max_k))))
    min_k = int(max(0, min(int(min_k), int(max_k))))

    if total_budget < t * min_k:
        min_k_eff = 0
    else:
        min_k_eff = min_k

    base = torch.full((t,), int(min_k_eff), device=curvature.device, dtype=torch.long)
    remaining = int(total_budget - int(base.sum().item()))
    if remaining <= 0:
        return base

    weights = curvature.float().clamp_min(0.0)
    if float(weights.sum().item()) <= 0.0:
        weights = torch.ones_like(weights)
    weights = weights / weights.sum().clamp_min(1e-6)

    raw = weights * float(remaining)
    extra = torch.floor(raw).to(torch.long)
    max_extra = max_k - min_k_eff
    if max_extra < 0:
        max_extra = 0
    extra = torch.minimum(extra, torch.full_like(extra, int(max_extra)))
    alloc = base + extra

    remaining = int(total_budget - int(alloc.sum().item()))
    if remaining > 0:
        frac = raw - torch.floor(raw)
        frac = frac.masked_fill(alloc >= max_k, -1.0)
        for _ in range(remaining):
            idx = torch.argmax(frac)
            if frac[idx].item() < 0:
                break
            alloc[idx] += 1
            if alloc[idx] >= max_k:
                frac[idx] = -1.0
    elif remaining < 0:
        over = -remaining
        order = torch.argsort(weights, descending=False)
        for idx in order:
            if over <= 0:
                break
            if alloc[idx] > min_k_eff:
                alloc[idx] -= 1
                over -= 1

    return alloc


def _v_cast_compress_frames(
    frames: torch.Tensor,  # [T, HW, D]
    deepstack_frames: List[torch.Tensor],  # list of [T, HW, D_ds]
    retain_ratio: float,
    min_k: int,
    tag: str = "",
) -> tuple[torch.Tensor, List[torch.Tensor], torch.Tensor]:
    t, tokens_per_frame, dim = frames.shape
    if t <= 0 or tokens_per_frame <= 0:
        keep_index = torch.zeros((0,), device=frames.device, dtype=torch.long)
        empty_main = frames.new_empty((0, dim))
        empty_ds = [ds.new_empty((0, ds.shape[-1])) for ds in deepstack_frames]
        return empty_main, empty_ds, keep_index

    if retain_ratio <= 0:
        retain_ratio = DEFAULT_RETAIN_RATIO

    frame_reps = F.normalize(frames.mean(dim=1), dim=-1, eps=1e-6)
    curvature = _compute_curvature(frame_reps)
    weights = torch.softmax(curvature.float() / float(DEFAULT_BUDGET_TEMP), dim=0)

    total_budget = int(round(float(t * tokens_per_frame) * float(retain_ratio)))
    total_budget = max(1, min(int(t * tokens_per_frame), total_budget))
    min_k = int(max(0, min(int(min_k), int(tokens_per_frame))))
    if min_k > 0:
        total_budget = max(total_budget, int(t * min_k))
    k_t = _allocate_budget_per_frame(
        weights,
        total_budget,
        min_k=min_k,
        max_k=tokens_per_frame,
    )

    tokens_out = []
    keep_indices = []
    deepstack_out = [[] for _ in deepstack_frames]

    for frame_idx in range(t):
        k = int(k_t[frame_idx].item())
        if k <= 0:
            continue

        frame_tokens = frames[frame_idx]
        rep = frame_reps[frame_idx]
        sim = F.cosine_similarity(frame_tokens, rep.unsqueeze(0), dim=-1, eps=1e-6)
        outlier = (1.0 - sim).float()
        norm = frame_tokens.float().norm(dim=-1)
        norm = (norm - norm.min()) / (norm.max() - norm.min() + 1e-6)
        score = outlier + norm
        topk_idx = torch.topk(score, k=k, largest=True, sorted=False).indices
        topk_idx, _ = torch.sort(topk_idx)

        tokens_out.append(frame_tokens.index_select(0, topk_idx))
        keep_indices.append(topk_idx + frame_idx * tokens_per_frame)

        for ds_idx, ds in enumerate(deepstack_frames):
            ds_frame = ds[frame_idx]
            deepstack_out[ds_idx].append(ds_frame.index_select(0, topk_idx))

    if len(tokens_out) == 0:
        keep_index = torch.zeros((0,), device=frames.device, dtype=torch.long)
        empty_main = frames.new_empty((0, dim))
        empty_ds = [ds.new_empty((0, ds.shape[-1])) for ds in deepstack_frames]
        return empty_main, empty_ds, keep_index

    tokens_comp = torch.cat(tokens_out, dim=0)
    keep_index = torch.cat(keep_indices, dim=0).to(torch.long)
    deepstack_comp = [
        torch.cat(chunks, dim=0) if len(chunks) > 0 else ds.new_empty((0, ds.shape[-1]))
        for chunks, ds in zip(deepstack_out, deepstack_frames)
    ]
    return tokens_comp, deepstack_comp, keep_index
# ───────────────────────────────────────────────────────────────────────────────────────────

# ══ Compressors for Qwen3-VL (16 temporal groups of merged tokens, 3-D positions, DeepStack) ══
# Every method returns a spec (the assembled prefix). Kept tokens keep their own 3-D positions and DeepStack features.
# (FlashVID, uniform pooling and random drop were run in the replication notebook.) Uniform pooling averages tokens, DeepStack features and positions over each window.

def _best_grid_hw(n, H, W):
    best = None
    for h in range(1, H + 1):
        for w in range(1, W + 1):
            key = (abs(h * w - n), abs(h / max(w, 1) - H / W), -h * w)
            if best is None or key < best[0]:
                best = (key, h, w)
    return best[1], best[2]

def compress_all(enc, seed, only=None, skip=()):
    G, P, Hm, Wm = enc["G"], enc["P"], enc["Hm"], enc["Wm"]
    X = enc["X"].view(G, P, -1)
    DS = [d.view(G, P, -1) for d in enc["DS"]]
    gp = _group_pos(enc)                                      # (3, G, P)
    dev = X.device
    jobs = []
    for r in METHOD_GRID.get("uniform", []):
        def un(r=r):
            h, w = _best_grid_hw(int(P * r), Hm, Wm)
            pool = lambda t: F.adaptive_avg_pool2d(t.float().reshape(G, Hm, Wm, -1).permute(0, 3, 1, 2), (h, w)) \
                .permute(0, 2, 3, 1).reshape(G, h * w, -1)
            fx = pool(X).to(X.dtype)
            fds = [pool(d).to(d.dtype) for d in DS]
            fpos = F.adaptive_avg_pool2d(gp.permute(1, 0, 2).reshape(G, 3, Hm, Wm).float(), (h, w)).round().long()
            fpos = fpos.reshape(G, 3, h * w).permute(1, 0, 2)
            s = assemble(enc, [(fx[g], [d[g] for d in fds], fpos[:, g]) for g in range(G)])
            s.update(inner=None, nvis=G * h * w)
            return s
        jobs.append(("uniform", str(r), un))
    for r in METHOD_GRID.get("random", []):
        def rd(r=r):
            k = int(P * r)
            gen = torch.Generator(device="cpu").manual_seed(seed)
            idx = torch.stack([torch.randperm(P, generator=gen)[:k].sort().values for _ in range(G)])
            groups = [(X[g][idx[g].to(dev)], [d[g][idx[g].to(dev)] for d in DS], gp[:, g][:, idx[g]]) for g in range(G)]
            s = assemble(enc, groups)
            s.update(inner=None, nvis=G * k)
            return s
        jobs.append(("random", str(r), rd))
    for r in METHOD_GRID.get("vcast", []):
        def vc(r=r):
            toks, dsc, keep = _v_cast_compress_frames(X, DS, retain_ratio=r, min_k=VCAST["min_k"])
            g_of = keep // P
            flat_pos = gp.reshape(3, G * P)
            groups = []
            for g in range(G):
                sel = g_of == g
                groups.append((toks[sel], [d[sel] for d in dsc], flat_pos[:, keep[sel].cpu()]))
            s = assemble(enc, groups)
            s.update(inner=None, nvis=int(toks.shape[0]))
            return s
        jobs.append(("vcast", str(r), vc))
    out, failures = {}, []
    for m, s, fn in jobs:
        if (only is not None and m not in only) or m in skip:
            continue
        try:
            out[(m, s)] = fn()
        except Exception as e:
            failures.append((m, s, f"{type(e).__name__}: {e}"))
    return out, failures

# ── self-test on a synthetic "video" (shapes, finiteness, budgets) ──
def _fake_enc(G=16, Hm=10, Wm=18, D=64, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    P = Hm * Wm
    base = torch.randn(1, P, D, generator=g)
    X = (base + torch.cumsum(0.25 * torch.randn(G, 1, D, generator=g), 0) + 0.6 * torch.randn(G, P, D, generator=g))
    pre = 10
    ids, pos, cur = [7] * pre, [list(range(pre))] * 3, pre
    ids, pos = list(ids), [list(p) for p in pos]
    for gi in range(G):
        ids += [5, VSTART_ID] + [VIDEO_TOKEN_ID] * P + [VEND_ID]
        for r in range(3):
            pos[r] += [cur, cur + 1]
        cur += 2
        hh, ww = torch.meshgrid(torch.arange(Hm), torch.arange(Wm), indexing="ij")
        pos[0] += [cur] * P; pos[1] += (cur + hh.flatten()).tolist(); pos[2] += (cur + ww.flatten()).tolist()
        cur += max(Hm, Wm)
        for r in range(3):
            pos[r] += [cur]
        cur += 1
    ids = torch.tensor(ids); pos = torch.tensor(pos)
    return {"ids": ids, "pos": pos, "vmask": ids == VIDEO_TOKEN_ID, "X": X.reshape(G * P, D).to(EMB_DEVICE, torch.float16),
            "DS": [torch.randn(G * P, D).to(EMB_DEVICE, torch.float16) for _ in range(N_DEEPSTACK)],
            "CA": torch.rand(G, P).to(EMB_DEVICE), "G": G, "Hm": Hm, "Wm": Wm, "P": P,
            "text_emb": torch.randn(len(ids), D).to(EMB_DEVICE, torch.float16), "next_pos": int(pos.max()) + 1,
            "n_full": G * P}

_enc = _fake_enc()
_comp, _fails = compress_all(_enc, seed=0)
assert not _fails, _fails
assert len(_comp) == sum(len(v) for v in METHOD_GRID.values()), "missing compressed passes"
for (m, s), c in _comp.items():
    assert torch.isfinite(c["embeds"].float()).all(), (m, s)
    assert c["embeds"].shape[0] == c["pos"].shape[1] == c["vmask"].shape[0], (m, s)
    assert int(c["vmask"].sum()) == c["nvis"] == c["ds"][0].shape[0], (m, s)
    print(f"  {m:9s} {s:5s} enters LLM with {c['nvis']:5d} visual tokens = {c['nvis'] / _enc['n_full']:.3f}"
          + (f" | inner {c['inner']['kind']} @ layer {c['inner']['layer']}" if c["inner"] else ""))
del _enc, _comp
print(f"Compressor self-test OK ✅  ({sum(len(v) for v in METHOD_GRID.values())} compressed passes)")

### 6 · Revision run

In [ ]:
# ── Main run: per video encode 3 frame sets, compress once, cache each prefix once, run every question on it ──
import zlib
done = set()
if os.path.exists(RES_PATH):
    with open(RES_PATH) as f:
        for line in f:
            try:
                done.add(json.loads(line)["qid"])
            except Exception:
                pass
print(f"{len(done)} questions already in results (resume)")

table = build_manifest()
T_START = time.time()
time_left = lambda: MAX_RUNTIME_HOURS * 3600 - (time.time() - T_START)

def video_groups():
    """Yields (video_key, local path, DataFrame of its questions still to run)."""
    if BENCHMARK == "videomme":
        finished = []                    # a chunk folder is deleted only once none of its videos can still be in use
        for ch in VIDEOMME_CONFIRM_CHUNKS:
            if time_left() < 1800:
                print("⏱️ Time budget reached — skipping remaining chunks"); return
            vids = videomme_fetch_chunk(ch)
            sub = table[(table.video_key.isin(vids.keys()) | table.video_alt.isin(vids.keys())) & ~table.qid.isin(done)]
            if QUICK_MAX_Q:
                sub = sub[sub.video_key.isin(sub.video_key.drop_duplicates().head(max(1, QUICK_MAX_Q // 3)))]
            if sub.empty:
                shutil.rmtree(os.path.join(WORK_DIR, f"vmme_{ch}"), ignore_errors=True)
                continue
            while len(finished) > 1:     # the main loop may still be reading the last video of the previous chunk
                shutil.rmtree(os.path.join(WORK_DIR, f"vmme_{finished.pop(0)}"), ignore_errors=True)
            print(f"Chunk {ch}: {len(sub)} questions, {sub.video_key.nunique()} videos")
            for vk, g in sub.groupby("video_key", sort=False):
                yield vk, vids.get(vk) or vids.get(g.video_alt.iloc[0]), g
            finished.append(ch)
            if QUICK_MAX_Q:
                return                                   # quick test: the first chunk with sample videos is enough
    else:
        sub = table[~table.qid.isin(done)]
        if QUICK_MAX_Q:
            per = max(1, QUICK_MAX_Q // max(1, sub.group.nunique()))
            sub = sub.groupby("group", group_keys=False).head(per).head(QUICK_MAX_Q)
        keys = sorted(sub.video_key.unique())
        print(f"{len(sub)} questions on {len(keys)} videos to run; downloading videos…")
        fetch = {"egoschema": egoschema_fetch, "longvideobench": longvideobench_fetch}[BENCHMARK]
        paths = fetch(keys)
        missing = [k for k in keys if k not in paths]
        for k in missing:
            with open(ERR_PATH, "a") as f:
                f.write(f"{k}\tmissing video\t{len(sub[sub.video_key == k])} questions dropped\n")
        if missing:
            print(f"⚠️ {len(missing)} videos could not be found/downloaded (listed in errors.log)")
        for vk, g in sub[sub.video_key.isin(paths.keys())].groupby("video_key", sort=False):
            yield vk, paths[vk], g

def _finish(res, row):
    res["correct"] = bool(res["pred"] == row.answer)
    return res

def run_video(fsets, rows, seed, first):
    enc = encode_frames(fsets["orig"], fsets["_meta"]["orig"])
    n_full = enc["n_full"]
    comp, fails = compress_all(enc, seed)
    if first:
        cache_check(enc, comp, rows)
    recs = {}
    for row in rows:
        recs[row.qid] = {"qid": row.qid, "video": row.video_key, "group": row.group, "task": row.task,
                         "gt": row.answer, "n_opt": min(len(row.options), len(LETTERS)), "n_full": n_full,
                         "noise": {}, "methods": {}}
    sufs = {row.qid: build_suffix_ids(row.question, row.options) for row in rows}
    # full model + option-order noise share one prefix
    P = prefill(full_spec(enc))
    for row in rows:
        n = recs[row.qid]["n_opt"]
        recs[row.qid]["full"] = _finish(suffix_pass(P, sufs[row.qid], n), row)
        for kind in NOISE_OPTION_ORDERS:
            res = suffix_pass(P, build_suffix_ids(row.question, reorder_options(row.options, kind)), n)
            recs[row.qid]["noise"][kind] = _finish(map_back(res, n, kind), row)
    del P
    # frame-shift noise
    for kind in NOISE_FRAME_SHIFTS:
        e = encode_frames(fsets[kind], fsets["_meta"][kind])
        P = prefill(full_spec(e))
        for row in rows:
            recs[row.qid]["noise"][kind] = _finish(suffix_pass(P, sufs[row.qid], recs[row.qid]["n_opt"]), row)
        del P, e
    for (m, s), c in comp.items():
        if c["inner"] is None:              # question-independent: cache the prefix
            P = prefill(c)
            outs = {row.qid: suffix_pass(P, sufs[row.qid], recs[row.qid]["n_opt"]) for row in rows}
            del P
        else:                               # FlashVID / HoliTom act inside the LLM and depend on the question
            outs = {row.qid: llm_pass(c, sufs[row.qid], recs[row.qid]["n_opt"]) for row in rows}
        for row in rows:
            res = _finish(outs[row.qid], row)
            res["retention"] = round(c["nvis"] / n_full, 4)
            res["retention_layer_avg"] = round(res["vis_mean_layers"] / n_full, 4)
            recs[row.qid]["methods"].setdefault(m, {})[s] = res
    return [recs[row.qid] for row in rows], fails

n_new, n_err, consec_fail, FIRST = 0, 0, 0, True
def _fail_guard():
    if consec_fail >= 6:
        raise RuntimeError("6 questions failed in a row — stopping instead of wasting GPU hours. Send me the log.")

reader = cf.ThreadPoolExecutor(1)
groups = video_groups()
nxt = next(groups, None)
fut = reader.submit(read_all_frame_sets, nxt[1]) if nxt else None
while nxt is not None:
    vk, path, g = nxt
    cur_fut = fut
    nxt = next(groups, None) if time_left() > 1200 else None
    if nxt is None and time_left() <= 1200:
        print("⏱️ Time budget reached — stopping after this video")
    fut = reader.submit(read_all_frame_sets, nxt[1]) if nxt else None       # decode the next video meanwhile
    t0 = time.time()
    rows = list(g.itertuples())
    try:
        fsets = cur_fut.result()
        if FIRST:
            equivalence_check(fsets, rows[0])
        recs, fails = run_video(fsets, rows, zlib.crc32(str(vk).encode()), FIRST)
        FIRST = False
    except RuntimeError as e:
        if "check failed" in str(e) or "Equivalence check" in str(e) or "Cache check" in str(e):
            raise
        n_err += len(rows); consec_fail += len(rows)
        with open(ERR_PATH, "a") as f: f.write(f"{vk}\tvideo\t{type(e).__name__}: {e}\n")
        print(f"  ✗ video {vk}: {type(e).__name__}: {e}"); _fail_guard(); continue
    except Exception as e:
        n_err += len(rows); consec_fail += len(rows)
        with open(ERR_PATH, "a") as f: f.write(f"{vk}\tvideo\t{type(e).__name__}: {e}\n")
        print(f"  ✗ video {vk}: {type(e).__name__}: {e}"); torch.cuda.empty_cache(); _fail_guard(); continue
    for m, s_, err in fails:
        with open(ERR_PATH, "a") as f: f.write(f"{vk}\tcompress {m}@{s_}\t{err}\n")
        print(f"  ! {m}@{s_} failed on video {vk} (skipped for this video): {err}")
    with open(RES_PATH, "a") as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
            done.add(r["qid"])
    n_new += len(recs); consec_fail = 0
    for r in recs:
        flips = sum(x["pred"] != r["full"]["pred"] for v in r["methods"].values() for x in v.values())
        nz = "".join("·" if x["pred"] == r["full"]["pred"] else "x" for x in r["noise"].values())
        print(f"  ✓ {r['qid']} [{r['group']}] gt={r['gt']} full={r['full']['pred']} noise[{nz}] "
              f"| {flips}/{sum(len(v) for v in r['methods'].values())} compressed changed")
    print(f"    video done in {time.time() - t0:.0f}s | {len(done)} questions total | {time_left() / 3600:.1f} h left")
    del fsets
    gc.collect(); torch.cuda.empty_cache()
reader.shutdown()
for _d in glob.glob(os.path.join(WORK_DIR, "vmme_*")):
    shutil.rmtree(_d, ignore_errors=True)
print(f"\nFinished: {n_new} new questions, {n_err} errors, {len(done)} total in {RES_PATH}")

### 7 · Summary and determinism check

In [ ]:
# ── Per-benchmark summary of the revision run (the full analysis is done offline with the main study's results) ──
R = []
with open(RES_PATH) as f:
    for line in f:
        try:
            R.append(json.loads(line))
        except Exception:
            pass
print(f"{len(R)} questions in {RES_PATH}")
summary = {"benchmark": BENCHMARK, "n": len(R), "model": MODEL_ID}
if R:
    ref = REFERENCE_FULL.get(BENCHMARK, {})
    both = [r for r in R if r["qid"] in ref]
    same = [r["full"]["pred"] == ref[r["qid"]][0] for r in both]
    gaps = [float(np.max(np.abs(np.array(r["full"]["logp"]) - np.array(ref[r["qid"]][1:])))) for r in both
            if len(r["full"]["logp"]) == len(ref[r["qid"]]) - 1]
    summary["reference_matched"] = len(both)
    summary["full_same_answer_as_main_study"] = float(np.mean(same)) if same else None
    summary["full_max_logp_gap"] = float(np.max(gaps)) if gaps else None
    summary["full_median_logp_gap"] = float(np.median(gaps)) if gaps else None
    full_acc = np.mean([r["full"]["correct"] for r in R])
    summary["full_acc"] = float(full_acc)
    summary["noise_flip_rate"] = {k: float(np.mean([r["noise"][k]["pred"] != r["full"]["pred"] for r in R if k in r["noise"]]))
                                  for k in list(NOISE_FRAME_SHIFTS) + NOISE_OPTION_ORDERS + globals().get("NOISE_PROMPTS", [])}
    summary["noise_acc"] = {k: float(np.mean([r["noise"][k]["correct"] for r in R if k in r["noise"]]))
                            for k in list(NOISE_FRAME_SHIFTS) + NOISE_OPTION_ORDERS + globals().get("NOISE_PROMPTS", [])}
    summary["vcast"] = {}
    for s_ in [str(x) for x in METHOD_GRID["vcast"]]:
        rr = [r for r in R if s_ in r["methods"].get("vcast", {})]
        if rr:
            acc = np.mean([r["methods"]["vcast"][s_]["correct"] for r in rr])
            summary["vcast"][s_] = {"n": len(rr), "flip_rate": float(np.mean([r["methods"]["vcast"][s_]["pred"] != r["full"]["pred"] for r in rr])),
                                    "acc": float(acc), "rel_acc": float(100 * acc / np.mean([r["full"]["correct"] for r in rr])),
                                    "retention": float(np.mean([r["methods"]["vcast"][s_]["retention"] for r in rr]))}
with open(os.path.join(OUT_DIR, f"summary_ext_{BENCHMARK}.json"), "w") as f:
    json.dump(summary, f, indent=1)
print(json.dumps(summary, indent=1))
if not QUICK_TEST and summary.get("full_same_answer_as_main_study") is not None and summary["full_same_answer_as_main_study"] < 0.99:
    print("⚠️ The full model's answers differ from the main study on more than 1% of questions. "
          "Send results.jsonl anyway; the offline analysis will use this run's own full pass where needed.")
print("\n➡️  Send only results.jsonl from", OUT_DIR)

## Next
Send `results.jsonl`. The V-CAST passes are merged with the replication results by question ID and analysed offline.